<a href="https://colab.research.google.com/github/kuds/mesozoic-labs/blob/main/notebooks/sb3_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mesozoic Labs — Stable-Baselines3 Training

Train one of the registered species through balance, locomotion, and a species-specific simulator task with PPO or SAC.

Pick a behavior in `BEHAVIOR` (a recipe label — `stand`, `walk`, `hunt` — or a deliverable's stage id) and the notebook walks that behavior's chain from the species' stage manifest, root first, within one session: each node REUSES a certified checkpoint from this run or from `TRUNK_FROM` where the reuse rule allows, is JUDGED if it was trained but never gated, and is TRAINED from its parent's handoff otherwise (`docs/BEHAVIOR_RECIPES_PLAN.md` §4.7). Later nodes consume the handoff records held in memory; files saved to Drive support analysis, archival and the cross-run reuse of certified ancestors, and the run bundle publishes every certified deliverable of the chain.

Change `SPECIES` in the configuration cell below. Stage budgets, environment settings, and algorithm hyperparameters are loaded from each species' `configs/<species>/stages.toml` manifest (or legacy `stage*.toml` files); those files and the generated species catalog are authoritative. Values differ by species and stage, so this notebook does not duplicate them.

For a smoke test, override the budget deliberately. For a full run, start with the configured values, measure memory and throughput on the target machine, and adjust parallel environments only from observed resource use. The repository does not publish a validated hardware-to-runtime or batch-size table.
Direction and terrain behaviors are available in `BEHAVIOR` for **all six registered species with PPO**. They use an explicit matched checkpoint/statistics pair and the selected species' `configs/<species>/behaviors/*.toml` recipes. Runs save checkpoints, evaluation summaries, videos and matching heat maps.


## 1. Setup & Installation
`REPO_REF` selects the code installed in Colab. Setup never discards checkout edits and refuses to switch code after repository modules have been imported; restart the runtime when changing revisions. Local notebooks use the current local checkout.


In [ ]:
# Install dependencies (Colab auto-detected; no-op locally)
import importlib
import os
import sys

REPO_REF = "main"  # @param {"type":"string"}
# For PR 540 before merge: codex/direction-and-random-terrain

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")

if IN_COLAB:
    # Configure headless rendering for MuJoCo (must happen before mujoco import)
    os.environ["MUJOCO_GL"] = "egl"
    NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
    if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
        os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
        with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
            f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

    # Legacy gym (and its shim packages) segfaults under NumPy 2.x on some
    # Colab images; purge it BEFORE the first stable_baselines3 import so
    # nothing can resolve to it. No-op when the packages are absent.
    get_ipython().system("pip uninstall -y -q gym gym-notices shimmy")

    # Install packages only if not already present. Every requirement is
    # QUOTED — an unquoted gymnasium>=0.29.0 made the shell treat ">" as a
    # redirect, installing an unpinned "gymnasium" and writing a junk file
    # named "=0.29.0" — and pinned to the exact tested set (the repo's
    # venv/CI versions) instead of open ranges.
    if any(importlib.util.find_spec(name) is None for name in ("mujoco", "gymnasium", "stable_baselines3", "mediapy", "matplotlib", "scipy", "imageio_ffmpeg")):
        get_ipython().system(
            'pip install -q "mujoco==3.10.0" "gymnasium==1.3.0" "stable-baselines3[extra]==2.9.0" mediapy matplotlib "scipy>=1.10.0" "imageio-ffmpeg>=0.5.1"'
        )
    import pathlib
    import subprocess

    repo_dir = pathlib.Path("/content/mesozoic-labs")
    if not REPO_REF.strip() or REPO_REF.startswith("-"):
        raise ValueError("REPO_REF must name a branch, tag, or commit.")
    fresh_checkout = not repo_dir.exists()
    if fresh_checkout:
        subprocess.run(["git", "clone", "--no-checkout", "https://github.com/kuds/mesozoic-labs.git", str(repo_dir)], check=True)
    if not (repo_dir / ".git").exists():
        raise RuntimeError(f"{repo_dir} is not a Git checkout; use a fresh runtime or a different directory.")
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=repo_dir, check=True)
    requested_commit = subprocess.check_output(["git", "rev-parse", "FETCH_HEAD^{commit}"], cwd=repo_dir, text=True).strip()
    current_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True).strip()
    if fresh_checkout or current_commit != requested_commit:
        if any(name == "environments" or name.startswith("environments.") for name in sys.modules):
            raise RuntimeError("REPO_REF changed after repository modules were imported. Restart the runtime, then rerun setup.")
        if not fresh_checkout and subprocess.check_output(["git", "status", "--porcelain"], cwd=repo_dir, text=True).strip():
            raise RuntimeError("The checkout has local edits. Save them before changing REPO_REF, or use a fresh runtime; no edits were discarded.")
        subprocess.run(["git", "checkout", "--detach", requested_commit], cwd=repo_dir, check=True)
    print(f"Repository ref: {REPO_REF}; commit: {requested_commit}")
    if importlib.util.find_spec("environments") is None:
        get_ipython().system("pip install -q -e /content/mesozoic-labs")
    print("Colab setup complete (EGL rendering enabled).")
else:
    print("Running locally; using this checkout. REPO_REF selects only the Colab checkout.")

In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Add repo root to path (works both locally from notebooks/ and in Colab)
if IN_COLAB:
    repo_root = Path("/content/mesozoic-labs")
else:
    repo_root = next(
        (path for path in (Path.cwd(), *Path.cwd().parents)
         if (path / "configs/species_manifest.toml").is_file()),
        None,
    )
    if repo_root is None:
        raise RuntimeError("Open this notebook from the mesozoic-labs checkout or its notebooks directory.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import gymnasium as gym
import mujoco

from environments.shared.config import load_all_stages, save_stage_config
from environments.shared.curriculum import load_vecnorm_stats
from environments.shared.evaluation import evaluate
from environments.shared.stage_manifest import load_stage_manifest, stage_dirname, stage_label

# Library imports — shared training infrastructure
from environments.shared.train_base import (
    create_vec_env,
)

print(f"MuJoCo version: {mujoco.__version__}")
print(f"Gymnasium version: {gym.__version__}")
print(f"Repo root: {repo_root}")

## 2. Configuration

Select a full species name in `SPECIES`. The notebook resolves it through the same registry as the SB3 CLI. Existing IDs and aliases also work. Config paths and saved runs retain stable IDs such as `trex`.

`BEHAVIOR` selects the deliverable to train (default `"hunt"`: stance → locomotion → behavior; `"stand"` on a species with a recovery stage is stance → recovery, whose frozen gate is then enforced). `TRUNK_FROM` names an earlier run (a run id under this species/algorithm log directory, or an absolute run directory) whose certified ancestors are reused instead of retrained — the target node itself is never reused across runs. `RETRAIN_FROM` names a chain node that must be trained here together with everything below it even when a certified copy exists; `RUN_LABEL` is a free-text label recorded beside every trained node's hyperparameter digest. A run directory that already records a node is never overwritten: a new variant is a new `RUN_ID`. `WIDEN_FROM` names an earlier run whose certified root checkpoint was trained behind this checkout's policy interface: its handoff pair is widened (zero columns for the new command dims, never re-trained) into this run's root stage directory and re-judged by the chain loop (`docs/BEHAVIOR_RECIPES_PLAN.md` §4.6); a widen session sets `SEED` to that run's seed. `WIDEN_MAX_REVISION_GAP` (default `1`: the Phase C bump alone) bounds how many policy-interface revisions behind that parent may be; the widen tool still checks the physics digest, `nq`/`nv`/`nu`, `action_dim` and the observation widths whatever the bound, so a larger gap only crosses fingerprint-only bumps — the two certified trex stance parents are r11 archives and need `2` (decision D-C17).

The six training selections include **Compsognathus Longipes** (anatomical proxy) and **Compsognathus Longipes (Robot)**. Each has separate PPO/SAC configs, normalization statistics and checkpoint identities. Their stages are balance, locomotion and non-contact target reaching. The robot keeps its fixed head and unpowered tail.

Compsognathus policies currently use simulator state and target coordinates with an MLP; the single head camera is available for rendering but is not a policy input. These are initial simulation recipes, not demonstrated walking policies or onboard controllers. Run the zero-action baseline below before tuning; stable standing alone is not evidence of learned locomotion. `QUICK_TEST` checks the pipeline and may stop at a curriculum gate.

Names come from `configs/species_manifest.toml`; see [species naming](../docs/SPECIES_NAMING.md) and the [Compsognathus training guide](../environments/compsognathus/README.md#training).

### Direction and terrain settings

Select any registered `SPECIES`, **ppo**, and a direction or terrain `BEHAVIOR`. Each species has nine TOML recipes: following direction, direction with speed and stops, terrain contact, slopes, bumps, depressions, mixed terrain, direction on slopes, and direction on mixed terrain (`combined_mixed_terrain`). Sizes and speeds are set for the selected species.

Set both source paths: `BEHAVIOR_LOAD_MODE="prepare"` takes this species' current-interface locomotion checkpoint plus its matched VecNormalize file; `"resume"` takes the exact behavior's saved pair and completes its remaining recipe budget; `"adapt"` takes a learned behavior pair into a compatible recipe. Resume/adapt require the matching `bundle.json` beside the model. For a periodic checkpoint, use `checkpoints/step-.../model.zip` and `vecnormalize.pkl` from the same directory. Species, source, config and identity mismatches are refused. The explicit source pair selects the parent; this route does not automatically train missing locomotion ancestors.

`BEHAVIOR_SEED=None` generates and records a fresh terrain/action seed for each new run. Use an integer to reproduce the layout. `BEHAVIOR_STEPS=None` uses the recipe budget; `QUICK_TEST=True` requests 4,096 additional steps. `BEHAVIOR_EVAL_ONLY=True` scores without learning. `BEHAVIOR_RECORD_VIDEO=True` saves a matching MP4, full/local heat maps, exact height grid and trajectory for every scored episode. These behaviors use **one CPU environment and the parent's PPO rollout settings**; canonical `N_ENVS` and `SEED` do not control them.

Each session saves under `<LOG_BASE>/<species>/ppo/behaviors/<behavior>/<BEHAVIOR_RUN_ID>`. Leave the id empty to generate one. Continuing a saved behavior uses a **new destination id** and explicit source pair. The canonical exploration, gate calibration, training chain, manual/resume cells and certified reports are skipped for these selections. Saved evaluations describe measured performance; they do not declare a curriculum gate passed.

An identical storage-cell rerun keeps its run ID and seed. Changing species, behavior or any run setting starts a fresh run and seed when those fields are blank/`None`; explicit `BEHAVIOR_RUN_ID` and `BEHAVIOR_SEED` take precedence. An occupied output directory is refused.


In [ ]:
# ===== SPECIES SELECTION =====
from environments.shared.species_names import species_display_name
from environments.shared.species_registry import get_species_config

SPECIES = "Velociraptor Mongoliensis"  # @param ["Velociraptor Mongoliensis", "Tyrannosaurus Rex", "Brachiosaurus Altithorax", "Dibothrosuchus Elaphros", "Compsognathus Longipes", "Compsognathus Longipes (Robot)"]

# Resolve before creating any log directories or provenance records.
SPECIES_CFG = get_species_config(SPECIES)
SPECIES = SPECIES_CFG.species
SPECIES_DISPLAY_NAME = species_display_name(SPECIES)

# ===== Training settings =====
ALGORITHM = "ppo"  # "ppo" or "sac"
N_ENVS = 4  # Canonical chain only; direction and terrain behaviors currently use one CPU environment
SEED = 42  # Canonical chain seed; direction and terrain behaviors use BEHAVIOR_SEED below
VERBOSE = 0  # 0=quiet, 1=progress bar, 2=debug
QUICK_TEST = False  # True = tiny run to verify setup works
USE_GOOGLE_DRIVE = True  # Mount Google Drive to save results
AUTO_DISCONNECT = True  # Disconnect the Colab runtime when training halts (gate failure or completion)

# ===== Behavior recipe (BEHAVIOR_RECIPES_PLAN §4.7) =====
BEHAVIOR = "hunt"  # @param ["stand", "walk", "hunt", "follow_direction", "follow_direction_speed", "terrain_contact", "sloped_terrain", "bumps_terrain", "depressions_terrain", "mixed_terrain", "combined_terrain", "combined_mixed_terrain"] {"allow-input":true}
TRUNK_FROM = ""  # optional earlier run whose certified ancestors to reuse
WIDEN_FROM = ""  # optional earlier run (id or absolute path) whose certified ROOT handoff is widened to this checkout's policy interface into RUN_DIR before the chain runs (BEHAVIOR_RECIPES_PLAN §4.6 Phase C)
WIDEN_MAX_REVISION_GAP = 1  # how many policy-interface revisions behind WIDEN_FROM's parent may be (D-C17); 1 = the Phase C bump alone; the two certified trex stance parents (r11) need 2 because r11 → r12 was fingerprint-only
# A widen session sets SEED above to the parent run's seed (D-C14) BEFORE the storage cell mints RUN_ID: the widen
# cell refuses any other value, and a run directory minted under the wrong seed is never re-minted (restart the runtime).
RETRAIN_FROM = ""  # optional chain node to train here with every node below it (empty = off; D-A19)
RUN_LABEL = ""  # optional free-text label recorded beside each trained node's hyperparameter digest (D-A21)

print(f"Species: {SPECIES_DISPLAY_NAME}")
print(f"Algorithm: {ALGORITHM.upper()}")
print(f"Quick test mode: {QUICK_TEST}")
print(f"Behavior: {BEHAVIOR}")
# ===== Direction and terrain behaviors (all species, PPO) =====
BEHAVIOR_LOAD_MODE = "prepare"  # @param ["prepare", "resume", "adapt"]
BEHAVIOR_CHECKPOINT = ""  # @param {"type":"string"}
BEHAVIOR_VECNORMALIZE = ""  # @param {"type":"string"}
BEHAVIOR_SEED = None  # None = fresh recorded terrain/action seed; or an integer from 0 through 2**32 - 1
BEHAVIOR_STEPS = None  # None = recipe budget (remaining budget for resume); an integer = additional steps
BEHAVIOR_EVAL_ONLY = False  # @param {"type":"boolean"}
BEHAVIOR_EVAL_EPISODES = 5  # @param {"type":"integer"}
BEHAVIOR_RECORD_VIDEO = True  # @param {"type":"boolean"}
BEHAVIOR_VIDEO_FPS = 25.0  # @param {"type":"number"}
# Set a new ID for another run in this kernel; an identical storage-cell rerun retains its ID/seed.
BEHAVIOR_RUN_ID = ""  # @param {"type":"string"}

COMMAND_TERRAIN_BEHAVIOR = BEHAVIOR in (
    "follow_direction", "follow_direction_speed", "terrain_contact", "sloped_terrain",
    "bumps_terrain", "depressions_terrain", "mixed_terrain", "combined_terrain", "combined_mixed_terrain",
)
if COMMAND_TERRAIN_BEHAVIOR:
    try:
        from environments.shared.behavior_notebook import validate_behavior_selection
    except ModuleNotFoundError as exc:
        raise RuntimeError(
            "This checkout does not contain direction and terrain notebook support. "
            "Select a current REPO_REF in setup and restart the runtime."
        ) from exc
    validate_behavior_selection(
        species=SPECIES, algorithm=ALGORITHM, behavior=BEHAVIOR,
        checkpoint=BEHAVIOR_CHECKPOINT, vecnormalize=BEHAVIOR_VECNORMALIZE, load_mode=BEHAVIOR_LOAD_MODE,
        trunk_from=TRUNK_FROM, widen_from=WIDEN_FROM, retrain_from=RETRAIN_FROM,
    )
    print("Behavior runtime: CPU, one environment, parent's PPO network/rollout settings.")
    print("N_ENVS and SEED above apply to the canonical chain; direction and terrain training uses BEHAVIOR_SEED.")


In [ ]:
# Direction and terrain training; the canonical curriculum uses the existing cells below.
# ===== BEHAVIOR STORAGE AND RUN PLAN =====
if COMMAND_TERRAIN_BEHAVIOR:
    from environments.shared.behavior_notebook import build_notebook_behavior_plan

    if USE_GOOGLE_DRIVE and IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        LOG_BASE = Path("/content/drive/MyDrive/mesozoic-labs/logs")
    else:
        LOG_BASE = repo_root / "logs"
        if USE_GOOGLE_DRIVE:
            print("Drive is available in Colab; this local run saves into the checkout's logs directory.")
    # Preserve the plan on an identical cell rerun; any changed selection
    # starts a fresh run/seed unless BEHAVIOR_RUN_ID/BEHAVIOR_SEED explicitly fix them.
    _behavior_selection = (
        SPECIES, ALGORITHM, BEHAVIOR, BEHAVIOR_LOAD_MODE, BEHAVIOR_CHECKPOINT, BEHAVIOR_VECNORMALIZE, BEHAVIOR_RUN_ID,
        BEHAVIOR_SEED, BEHAVIOR_STEPS, QUICK_TEST, BEHAVIOR_EVAL_ONLY, BEHAVIOR_EVAL_EPISODES,
        BEHAVIOR_RECORD_VIDEO, BEHAVIOR_VIDEO_FPS, str(LOG_BASE),
    )
    _same_behavior_selection = _behavior_selection == globals().get("_ACTIVE_BEHAVIOR_SELECTION")
    _behavior_id = BEHAVIOR_RUN_ID or (globals().get("_ACTIVE_BEHAVIOR_RUN_ID", "") if _same_behavior_selection else "")
    _behavior_seed = BEHAVIOR_SEED
    if _behavior_seed is None and _same_behavior_selection:
        _behavior_seed = globals().get("_ACTIVE_BEHAVIOR_SEED")
    BEHAVIOR_PLAN = build_notebook_behavior_plan(
        repo_root=repo_root, log_base=LOG_BASE, species=SPECIES, algorithm=ALGORITHM,
        behavior=BEHAVIOR, checkpoint=BEHAVIOR_CHECKPOINT, vecnormalize=BEHAVIOR_VECNORMALIZE,
        load_mode=BEHAVIOR_LOAD_MODE, seed=_behavior_seed, steps=BEHAVIOR_STEPS,
        quick_test=QUICK_TEST, eval_only=BEHAVIOR_EVAL_ONLY, eval_episodes=BEHAVIOR_EVAL_EPISODES,
        record_video=BEHAVIOR_RECORD_VIDEO, video_fps=BEHAVIOR_VIDEO_FPS, run_id=_behavior_id,
    )
    _ACTIVE_BEHAVIOR_SELECTION = _behavior_selection
    _ACTIVE_BEHAVIOR_RUN_ID = BEHAVIOR_PLAN.output_dir.name
    _ACTIVE_BEHAVIOR_SEED = BEHAVIOR_PLAN.seed
    CHAIN = []  # The canonical chain has no nodes in a direction or terrain session.
    print(f"Behavior: {BEHAVIOR}; load mode: {BEHAVIOR_PLAN.load_mode}")
    print(f"Actual behavior seed: {BEHAVIOR_PLAN.seed}; CPU; one environment")
    print(f"Recipe: {BEHAVIOR_PLAN.recipe_path}")
    print(f"Source model: {BEHAVIOR_PLAN.checkpoint_path}")
    print(f"Source normalization: {BEHAVIOR_PLAN.vecnormalize_path}")
    print(f"Output directory: {BEHAVIOR_PLAN.output_dir}")
    print(f"Additional steps: {BEHAVIOR_PLAN.steps if BEHAVIOR_PLAN.steps is not None else 'recipe budget / remaining on resume'}")
    print(f"Evaluation only: {BEHAVIOR_PLAN.eval_only}; episodes: {BEHAVIOR_PLAN.eval_episodes}; video/maps: {BEHAVIOR_PLAN.record_video}")


In [ ]:
if not globals().get("COMMAND_TERRAIN_BEHAVIOR", False):
    # ============================================================
    # Storage Configuration
    # ============================================================
    # When USE_GOOGLE_DRIVE is True and running in Colab, logs and
    # models are saved to Google Drive so they persist across sessions.
    # Otherwise, everything is saved to the local filesystem.

    if USE_GOOGLE_DRIVE and IN_COLAB:
        from google.colab import drive

        drive.mount("/content/drive")
        LOG_BASE = Path("/content/drive/MyDrive/mesozoic-labs/logs")
        LOG_BASE.mkdir(parents=True, exist_ok=True)
        print(f"Google Drive mounted. Logs will be saved to: {LOG_BASE}")
    elif USE_GOOGLE_DRIVE and not IN_COLAB:
        print("Warning: USE_GOOGLE_DRIVE is True but not running in Colab. Using local storage.")
        LOG_BASE = repo_root / "logs"
        print(f"Logs will be saved to: {LOG_BASE}")
    else:
        LOG_BASE = repo_root / "logs"
        print(f"Logs will be saved to: {LOG_BASE}")

    # Create one run directory for the current in-memory chain. Re-running this
    # cell in the same runtime reuses it; certified nodes of an EARLIER run come
    # in through TRUNK_FROM (below), never by pointing RUN_ID at that run.
    RUN_ID = ""
    if not RUN_ID:
        RUN_ID = globals().get("_ACTIVE_RUN_ID", "")
    if not RUN_ID:
        RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
    _ACTIVE_RUN_ID = RUN_ID
    RUN_DIR = LOG_BASE / SPECIES / ALGORITHM.lower() / RUN_ID
    RUN_DIR.mkdir(parents=True, exist_ok=True)

    # Capture immutable code, runtime, seed, and plant provenance before training.
    # Re-running this cell is safe when the identifying settings are unchanged.
    from environments.shared.constants import PUBLICATION_SEED_START
    from environments.shared.plant_contract import current_plant_identity
    from environments.shared.result_bundle import initialize_result_bundle, validate_result_bundle

    CHECKPOINT_SELECTION_SEED = SEED + 1000
    EVALUATION_SEED = SEED + 3000
    HARDWARE_LABEL = "Google Colab" if IN_COLAB else "local"
    PLANT_IDENTITY = current_plant_identity(SPECIES)
    PROVENANCE_PATH = initialize_result_bundle(
        RUN_DIR,
        species=SPECIES,
        algorithm=ALGORITHM,
        backend="stable-baselines3",
        seed=SEED,
        evaluation_seeds=[CHECKPOINT_SELECTION_SEED, EVALUATION_SEED],
        seed_roles={
            "training": SEED,
            "checkpoint_selection_evaluation": CHECKPOINT_SELECTION_SEED,
            "publication_evaluation": EVALUATION_SEED,
            # The 40-seed certification panel (3042-3081) the stance report and the
            # recovery resolver both roll from; the audit binds their evidence to it
            # (BEHAVIOR_RECIPES_PLAN §4.5, decision D-B17).
            "certification_panel": PUBLICATION_SEED_START,
        },
        evaluation_episodes=30,
        parallel_envs=N_ENVS,
        hardware=HARDWARE_LABEL,
        plant_identity=PLANT_IDENTITY.to_dict(),
        run_id=RUN_ID,
        repository_root=repo_root,
    )
    print(f"Run directory: {RUN_DIR}")
    print(f"Provenance:    {PROVENANCE_PATH}")

    # TRUNK_FROM: an earlier run whose certified ancestors the chain loop may reuse
    # (BEHAVIOR_RECIPES_PLAN §4.2). A run id resolves under this species/algorithm
    # log directory; an absolute path names a run directory anywhere. It must be a
    # finished bundle of the SAME species, algorithm and backend, and never this run.
    from environments.shared.result_bundle import ResultBundleError, canonical_algorithm, load_provenance

    TRUNK_DIR = None
    if TRUNK_FROM:
        TRUNK_DIR = Path(TRUNK_FROM)
        if not TRUNK_DIR.is_absolute():
            TRUNK_DIR = LOG_BASE / SPECIES / ALGORITHM.lower() / TRUNK_FROM
        if TRUNK_DIR.resolve() == RUN_DIR.resolve():
            raise RuntimeError(f"TRUNK_FROM={TRUNK_FROM!r} is this run's own directory; a trunk is an EARLIER run")
        try:
            _trunk_provenance = load_provenance(TRUNK_DIR)
        except ResultBundleError as exc:
            raise RuntimeError(
                f"TRUNK_FROM={TRUNK_FROM!r} ({TRUNK_DIR}) is not a run with a provenance.json, so nothing in it "
                f"can be a certified ancestor: {exc}"
            ) from exc
        _trunk_identity = {
            "species": _trunk_provenance.get("species"),
            "algorithm": _trunk_provenance.get("algorithm"),
            "backend": _trunk_provenance.get("backend"),
        }
        _this_identity = {"species": SPECIES, "algorithm": canonical_algorithm(ALGORITHM), "backend": "stable-baselines3"}
        if _trunk_identity != _this_identity:
            raise RuntimeError(
                f"TRUNK_FROM={TRUNK_FROM!r} was trained as {_trunk_identity}, not {_this_identity}; a certified "
                "ancestor must come from the same species, algorithm and backend"
            )
        print(f"Trunk run:     {TRUNK_DIR} (run_id {_trunk_provenance.get('run_id')!r})")

## 3. Explore the Environment

In [ ]:
if not globals().get("COMMAND_TERRAIN_BEHAVIOR", False):
    # Use the shared registry config already resolved in the selection cell.
    EnvClass = SPECIES_CFG.env_class

    # Stage files and ordering are resolved by each species' stage manifest.
    STAGE_CONFIGS = load_all_stages(SPECIES)

    # The behavior chain is resolved ONCE, here, through the manifest: BEHAVIOR
    # names a deliverable node (a recipe label resolves to the deepest deliverable
    # carrying it, an id to itself) and CHAIN is that node's ancestors root-first,
    # then the node. The chain loop (section 5) walks CHAIN in this order.
    MANIFEST = load_stage_manifest(SPECIES)
    TARGET_NODE = MANIFEST.resolve_behavior(BEHAVIOR)
    CHAIN = MANIFEST.chain_for(TARGET_NODE.id)
    # RETRAIN_FROM (D-A19) names a chain node by its manifest id: it and every
    # node below it train here even when a certified copy exists.
    RETRAIN_NODE = MANIFEST.resolve(RETRAIN_FROM) if RETRAIN_FROM else None
    if RETRAIN_NODE is not None and RETRAIN_NODE not in CHAIN:
        raise RuntimeError(
            f"RETRAIN_FROM={RETRAIN_FROM!r} is not on the chain of behavior {BEHAVIOR!r} "
            f"({[node.id for node in CHAIN]}); name one of its nodes or leave it empty"
        )

    env = EnvClass()
    try:
        print(f"Species: {SPECIES_DISPLAY_NAME}")
        print(f"Environment: {EnvClass.__name__}")
        print(f"Observation space: {env.observation_space.shape}")
        print(f"Action space: {env.action_space.shape}")
        print(f"Number of actuators: {env.model.nu}")
    finally:
        env.close()

    print(f"\nBehavior {BEHAVIOR!r} -> deliverable {TARGET_NODE.id!r}; chain: {' -> '.join(node.id for node in CHAIN)}")
    if RETRAIN_NODE is not None:
        print(f"RETRAIN_FROM {RETRAIN_NODE.id!r}: it and every node below it train here (no reuse)")
    print(f"\n  {'#':>2}  {'id':<12} {'name':<30} {'parent':<12} {'recipe':<8} {'deliverable':<12} gate")
    for entry in MANIFEST.stages:
        cfg = STAGE_CONFIGS[entry.reference]
        mark = "*" if entry in CHAIN else " "
        gate = cfg.get("curriculum_kwargs", {}).get("gate_kind", "?")
        print(
            f"{mark} {entry.position:>2}  {entry.id:<12} {cfg['name']:<30} {entry.warm_start_from or '-':<12} "
            f"{entry.recipe or '-':<8} {'yes' if entry.deliverable else 'no':<12} {gate}"
        )
    print("  (* = on this behavior's chain)")

In [ ]:
if not globals().get("COMMAND_TERRAIN_BEHAVIOR", False):
    # Widen an earlier run's certified root checkpoint (BEHAVIOR_RECIPES_PLAN §4.6)
    # WIDEN_FROM names an earlier run — a run id under this species/algorithm log
    # directory, or an absolute run directory — whose certified ROOT handoff was
    # trained at most WIDEN_MAX_REVISION_GAP policy-interface revisions behind
    # this checkout (D-C17; the default 1 is the Phase C bump alone: the tool's
    # gate still checks the physics digest, nq/nv/nu, action_dim and the widths
    # whatever the bound, so a larger gap crosses fingerprint-only bumps only —
    # the certified trex r11 stance parents need 2). Its checkpoint
    # pair is widened (zero columns for the new command dims; never re-labelled,
    # never re-trained; decisions D-C8..D-C12) into THIS run's root stage
    # directory as a judge-ready node: the chain loop below finds no verdict
    # there, refuses to reuse it — loudly — and JUDGES the <stage_label>_final.*
    # pair the tool wrote, re-rolling the certification panel under the current
    # plant. A widened node is a root of this run (WIDEN_LINEAGE_KEYS, never
    # LOAD_LINEAGE_KEYS), so it is brought in here, in a NEW RUN_ID — never by
    # pointing RUN_ID at the old run (D-C13). SEED must equal the parent's seed
    # (D-C14): the provenance minted above publishes training_seed = SEED, and
    # seed replication counts distinct seeds. With WIDEN_FROM empty this cell
    # does nothing. It runs before the training infrastructure, so it imports
    # every name it uses itself.
    import json
    from pathlib import Path

    from environments.shared.config import refuse_occupied_stage_dir
    from environments.shared.result_bundle import ResultBundleError, canonical_algorithm, load_provenance
    from environments.shared.scripts.widen_checkpoint import widen_checkpoint
    from environments.shared.stage_manifest import stage_dir_candidates, stage_dirname

    if not WIDEN_FROM:
        print("WIDEN_FROM is empty: no earlier checkpoint is widened into this run.")
    else:
        WIDEN_DIR = Path(WIDEN_FROM)
        if not WIDEN_DIR.is_absolute():
            WIDEN_DIR = LOG_BASE / SPECIES / ALGORITHM.lower() / WIDEN_FROM
        if WIDEN_DIR.resolve() == RUN_DIR.resolve():
            raise RuntimeError(f"WIDEN_FROM={WIDEN_FROM!r} is this run's own directory; a widen parent is an EARLIER run")
        try:
            _widen_provenance = load_provenance(WIDEN_DIR)
        except ResultBundleError as exc:
            raise RuntimeError(
                f"WIDEN_FROM={WIDEN_FROM!r} ({WIDEN_DIR}) is not a run with a provenance.json, so it holds no "
                f"certified root to widen: {exc}"
            ) from exc
        _widen_identity = {
            "species": _widen_provenance.get("species"),
            "algorithm": _widen_provenance.get("algorithm"),
            "backend": _widen_provenance.get("backend"),
        }
        _widen_this_identity = {
            "species": SPECIES,
            "algorithm": canonical_algorithm(ALGORITHM),
            "backend": "stable-baselines3",
        }
        if _widen_identity != _widen_this_identity:
            raise RuntimeError(
                f"WIDEN_FROM={WIDEN_FROM!r} was trained as {_widen_identity}, not {_widen_this_identity}; a widened "
                "root must come from the same species, algorithm and backend"
            )
        _widen_parent_run_id = _widen_provenance.get("run_id") or WIDEN_DIR.name

        # The chain's ROOT is the node widened: everything below it trains here
        # from the re-judged handoff (or is reused through TRUNK_FROM).
        _widen_root = CHAIN[0]
        _widen_source = next(
            (WIDEN_DIR / name for name in stage_dir_candidates(SPECIES, _widen_root.reference) if (WIDEN_DIR / name).is_dir()),
            None,
        )
        if _widen_source is None:
            raise RuntimeError(
                f"WIDEN_FROM={WIDEN_FROM!r} ({WIDEN_DIR}) holds no {_widen_root.id!r} stage directory "
                f"({' / '.join(stage_dir_candidates(SPECIES, _widen_root.reference))}); nothing to widen"
            )
        # D-C14: the widened root inherits the parent's run block, and this run's
        # provenance publishes training_seed = SEED — the two must agree, or seed
        # replication would count a seed the weights were never trained under.
        if not (_widen_source / "stage_config.json").is_file():
            raise RuntimeError(f"{_widen_source} records no stage_config.json (no run block to inherit); nothing to widen")
        _widen_parent_run = json.loads((_widen_source / "stage_config.json").read_text(encoding="utf-8")).get("run") or {}
        _widen_parent_seed = _widen_parent_run.get("seed")
        if _widen_parent_seed is None:
            raise RuntimeError(
                f"{_widen_source / 'stage_config.json'} records no run seed; the widened root inherits the parent's "
                "run facts and cannot invent them"
            )
        if _widen_parent_seed != SEED:
            raise ValueError(
                f"SEED={SEED} but the widen parent {_widen_parent_run_id!r} trained {_widen_root.id!r} under "
                f"seed {_widen_parent_seed}; a widened run publishes the parent's seed as its own (D-C14). Set "
                f"SEED = {_widen_parent_seed} in the configuration cell, then restart the runtime (or "
                f"`del _ACTIVE_RUN_ID`) before Run all: the storage cell already minted {RUN_DIR} with "
                f"training_seed = {SEED} and refuses to re-mint that directory under another seed, so the corrected "
                "session needs a fresh RUN_ID (delete the stray directory; it holds only provenance.json)"
            )

        # D-A20 / D-C13: a stage directory that already records a node is never
        # overwritten; a re-run of this cell in the same RUN_DIR refuses here, and
        # a second widening is a new RUN_ID.
        _widen_target = RUN_DIR / stage_dirname(SPECIES, _widen_root.reference)
        refuse_occupied_stage_dir(_widen_target, task_load_mode=None)
        print(f"Widening {_widen_root.id!r} from run {_widen_parent_run_id!r}: {_widen_source} -> {_widen_target}")
        _widen_result = widen_checkpoint(
            species=SPECIES,
            stage=_widen_root.reference,
            parent_stage_dir=_widen_source,
            target_stage_dir=_widen_target,
            algorithm=ALGORITHM,
            label=RUN_LABEL or None,
            parent_run_id=_widen_parent_run_id,
            max_revision_gap=WIDEN_MAX_REVISION_GAP,
        )
        _widen_report = _widen_result.report
        print(f"Widened {_widen_root.id!r} ({_widen_result.algorithm.upper()}) into {_widen_result.target_stage_dir}")
        print(f"  observation_dim: {_widen_result.from_observation_dim} -> {_widen_result.to_observation_dim}")
        print(f"  policy-interface revisions crossed: {_widen_result.revision_gap} (WIDEN_MAX_REVISION_GAP={WIDEN_MAX_REVISION_GAP})")
        print(f"  handoff: {_widen_result.handoff_name} ({_widen_result.model_zip.name} + {_widen_result.vecnorm_pkl.name})")
        print(f"  final copies: {_widen_result.final_zip.name} + {_widen_result.final_vecnorm_pkl.name}")
        print(f"  padded tensors: {', '.join(_widen_report['padded_tensors'])}")
        print(f"  max |padded column|: {_widen_report['max_padded_column_abs']}")
        print(
            f"  max action delta: {_widen_result.max_action_delta_zero_command:.3g} (zero command) / "
            f"{_widen_result.max_action_delta_probe_command:.3g} (probe command); tolerance {_widen_report['action_delta_atol']}"
        )
        print(f"  num_timesteps inherited: {_widen_report['num_timesteps']:,}; report: {_widen_result.report_path}")
        print(f"  The chain loop will JUDGE this node (no verdict yet); parent run {_widen_parent_run_id!r} is untouched.")

In [ ]:
if not globals().get("COMMAND_TERRAIN_BEHAVIOR", False):
    # Run random episodes to establish a baseline
    env = EnvClass()
    n_episodes = 5
    episode_rewards = []
    episode_lengths = []

    for ep in range(n_episodes):
        obs, info = env.reset(seed=ep)
        total_reward = 0
        step = 0
        while True:
            action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            step += 1
            if terminated or truncated:
                break
        episode_rewards.append(total_reward)
        episode_lengths.append(step)
        print(f"  Episode {ep + 1}: reward={total_reward:.2f}, length={step}")

    print("\nRandom policy baseline:")
    print(f"  Avg reward: {np.mean(episode_rewards):.2f} +/- {np.std(episode_rewards):.2f}")
    print(f"  Avg length: {np.mean(episode_lengths):.1f} +/- {np.std(episode_lengths):.1f}")
    env.close()

### 3b. Pre-flight: zero-action baseline

The cell above scores a **random** policy, which any trained policy beats trivially. The
floor that actually decides whether stage 1 learned anything is the **do-nothing** policy.

Actions are residuals around each species' home keyframe (`home-keyframe-residual/v1`), so
`action = 0` commands the nominal stance exactly. On a passively stable plant that is a real
policy, and on a balance stage it can be a strong one — strong enough that Tyrannosaurus Rex runs
`20260723_204941` and `20260724_140441` both scored *below* it and advanced anyway, because
the gate at the time was `min_avg_reward = 100.0`.

Two numbers come out, and they answer different questions:

| number | meaning |
|---|---|
| **reward** | unconditional mean, including episodes the statue falls out of. A gate below this is cleared by doing nothing. |
| **reward standing** | conditioned on reaching the horizon. A gate between the two is cleared by a policy that has learned only "do not fall". |

`configs/trex/stance.toml` asks for exactly this check: *"this is a measured constant
and will go stale if the plant or the reward weights change — re-run the baseline script and
update it, or better, wire that script in as an automatic pre-stage check."* This cell is that
check. Results are written to Drive so the calibration travels with the run.

By default this measures only the species this notebook is training (~1 min) and writes the
record to that species' own log directory:

```
<LOG_BASE>/<species>/zero_action_baselines/<timestamp>.json   # the numbers, machine-readable
<LOG_BASE>/<species>/zero_action_baselines/<timestamp>.txt    # the table printed below
<RUN_DIR>/zero_action_baseline.json                           # copy, so the run carries its calibration
```

The JSON records the plant identity and the stage-1 env kwargs alongside the numbers, because
those are what make the constant go stale. Widen `BASELINE_SPECIES` to all four (~4 min) to
find out whether a weak gate is species-specific or a shared-template problem — each species'
record still lands in its own directory.

---

To run the same thing from a **Colab terminal** instead of the notebook:

```bash
cd /content
[ -d mesozoic-labs ] || git clone https://github.com/kuds/mesozoic-labs.git
cd mesozoic-labs

# Debian's patched setuptools breaks legacy setup.py sdists; upgrade it first.
pip install -q -U setuptools
pip install -q -e .          # core deps only — the baseline needs no torch/SB3

SPECIES=trex   # or velociraptor / brachiosaurus / dibothrosuchus
OUT=/content/drive/MyDrive/mesozoic-labs/logs/$SPECIES/zero_action_baselines
[ -d /content/drive/MyDrive ] || echo "WARNING: Drive not mounted — saving locally only"
mkdir -p "$OUT"

python environments/shared/scripts/zero_action_baseline.py "$SPECIES" \
    --episodes 40 --seed 3042 2>&1 | tee "$OUT/$(date +%Y%m%d_%H%M%S)_terminal.txt"
```

Drive is only visible to the terminal once the notebook has mounted it (cell 2 of section 1).

Pass several species names to measure them in one go — the transcript then covers all of
them, so `tee` it wherever makes sense rather than into one species' directory. Add
`--sweep-noise` to sweep `reset_noise_scale` instead of measuring a single point; that is
how the stage-1 reset noise gets calibrated.

In [ ]:
if not globals().get("COMMAND_TERRAIN_BEHAVIOR", False):
    # ============================================================
    # Pre-flight: zero-action baseline (stage-1 gate calibration)
    # ============================================================
    # Scores the do-nothing policy on stage 1 and checks each species'
    # min_avg_reward gate against it.  These are measured constants: they go stale
    # whenever the plant or the stage-1 reward weights change, so the result is
    # saved to Drive with the plant identity and the env kwargs that produced it.

    import json

    from environments.shared.config import load_stage_config
    from environments.shared.constants import PUBLICATION_SEED_START
    from environments.shared.plant_contract import current_plant_identity
    from environments.shared.scripts.zero_action_baseline import build_env, gate_margin, score
    from environments.shared.species_names import resolve_species_id, species_display_name

    # Just the species this notebook is training (~1 min).  To find out whether a weak
    # gate is species-specific or a shared-template problem, widen this to all four
    # (~4 min) -- each species' record still lands in its own log directory:
    #   from environments.shared.species_names import species_display_names
    #   BASELINE_SPECIES = list(species_display_names())
    BASELINE_SPECIES = [SPECIES]
    BASELINE_STAGE = 1
    BASELINE_EPISODES = 40  # matches the figures recorded in the stage-1 configs (configs/*/, manifest-named)
    BASELINE_SEED = PUBLICATION_SEED_START  # the publication_evaluation seed role

    results = {}
    for _species_input in BASELINE_SPECIES:
        _species = resolve_species_id(_species_input)
        _env = build_env(_species, BASELINE_STAGE)
        try:
            _result = score(_env, BASELINE_EPISODES, BASELINE_SEED)
        finally:
            _env.close()

        _stage_cfg = load_stage_config(_species, BASELINE_STAGE)
        _gate = _stage_cfg["curriculum_kwargs"].get("min_avg_reward")

        # A gate only means something if it sits above the floor a statue reaches.
        if _gate is None:
            _verdict = "NO GATE"
        elif _gate <= _result["reward_mean"]:
            _verdict = "FAILS — a statue clears this gate"
        elif _result["n_standing"] == 0:
            # The statue never reaches the horizon, so there is no standing floor to
            # compare against.  That is its own problem: a plant that falls over under
            # its own home controller is not ready for a balance stage.
            _verdict = "CHECK PLANT — the statue never survives a full episode"
        elif _gate <= _result["reward_mean_standing"]:
            _verdict = "WEAK — binds only against a falling statue"
        else:
            _verdict = "OK"

        _result.update(
            {
                "species": _species,
                "display_name": species_display_name(_species),
                "stage": BASELINE_STAGE,
                "min_avg_reward": _gate,
                "verdict": _verdict,
                "margin_over_mean": gate_margin(_gate, _result["reward_mean"]),
                "margin_over_standing": gate_margin(_gate, _result["reward_mean_standing"]),
                "env_kwargs": _stage_cfg["env_kwargs"],
                "plant_identity": current_plant_identity(_species).to_dict(),
            }
        )
        results[_species] = _result

    # ---- table ----------------------------------------------------------------
    _header = (
        f"{'species':<30}{'reward':>10}{'mean-std':>10}{'standing':>10}"
        f"{'full-hz':>9}{'gate':>9}  verdict"
    )
    _lines = [
        f"zero-action baseline — stage {BASELINE_STAGE}, {BASELINE_EPISODES} episodes, seed {BASELINE_SEED}",
        "",
        _header,
        "-" * len(_header),
    ]
    for _species, _r in results.items():
        _gate_text = "—" if _r["min_avg_reward"] is None else f"{_r['min_avg_reward']:.0f}"
        _standing_text = "—" if _r["n_standing"] == 0 else f"{_r['reward_mean_standing']:.1f}"
        _lines.append(
            f"{species_display_name(_species):<30}{_r['reward_mean']:>10.1f}{_r['reward_mean_minus_std']:>10.1f}"
            f"{_standing_text:>10}{_r['full_horizon_share']:>8.0%}"
            f"{_gate_text:>9}  {_r['verdict']}"
        )
    _lines += [
        "",
        "A trained stage-1 policy must beat 'reward', 'mean-std' AND 'full-hz' to have",
        "learned to balance at all, and 'standing' to have learned more than 'do not fall'.",
    ]
    _report_text = "\n".join(_lines)
    print(_report_text)

    # ---- save to the log directory --------------------------------------------
    # One record per species, under that species' own log dir, so a baseline sits
    # next to the runs it calibrates:
    #   <LOG_BASE>/<species>/zero_action_baselines/<timestamp>.json
    _log_base = globals().get("LOG_BASE")
    if _log_base is not None:
        _stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        _captured_at = datetime.now().isoformat()

        def _payload_for(species_names):
            return {
                "schema": "mesozoic.zero-action-baseline/v1",
                "captured_at": _captured_at,
                "stage": BASELINE_STAGE,
                "episodes": BASELINE_EPISODES,
                "seed": BASELINE_SEED,
                "results": {name: results[name] for name in species_names},
            }

        for _species in results:
            _species_dir = Path(_log_base) / _species / "zero_action_baselines"
            _species_dir.mkdir(parents=True, exist_ok=True)
            (_species_dir / f"{_stamp}.json").write_text(
                json.dumps(_payload_for([_species]), indent=2, sort_keys=True, allow_nan=False)
            )
            print(f"\nSaved: {_species_dir / (_stamp + '.json')}")

        # The human-readable table covers every species measured, so it belongs with
        # the one this notebook is training.
        _table_dir = Path(_log_base) / SPECIES / "zero_action_baselines"
        _table_dir.mkdir(parents=True, exist_ok=True)
        (_table_dir / f"{_stamp}.txt").write_text(_report_text + "\n")

        # Drop a copy in the run directory too, so this run carries the calibration
        # its own gate was judged against.
        _run_dir = globals().get("RUN_DIR")
        if _run_dir is not None and SPECIES in results:
            (Path(_run_dir) / "zero_action_baseline.json").write_text(
                json.dumps(_payload_for([SPECIES]), indent=2, sort_keys=True, allow_nan=False)
            )
            print(f"Copied to run: {Path(_run_dir) / 'zero_action_baseline.json'}")
    else:
        print("\nLOG_BASE not defined — run the storage-configuration cell to save results.")

## 4. Training Infrastructure

In [ ]:
import time

from stable_baselines3 import PPO, SAC

from environments.shared.config import read_stage_duration, record_stage_duration, refuse_occupied_stage_dir
from environments.shared.evaluation import eval_policy
from environments.shared.plant_contract import validate_model_plant
from environments.shared.replication import discover_replicates_for_run
from environments.shared.reporting import (
    build_stage_results_from_eval_data,
    generate_stage_artifacts,
)
from environments.shared.reporting import (
    save_evaluation_episodes as _lib_save_evaluation_episodes,
)
from environments.shared.reporting import (
    save_result_bundle as _lib_save_result_bundle,
)
from environments.shared.reporting import (
    write_training_summary as _lib_write_training_summary,
)
from environments.shared.task_fingerprint import (
    derive_stage_task_fingerprint,
    read_checkpoint_task_fingerprint,
    validate_declared_parent,
)
from environments.shared.train_base import (
    _build_core_callbacks,
    _create_or_load_model,
    _ensure_sb3,
    _is_resume_continuation,
    _maybe_ent_coef_decay_callback,
    _prepare_alg_kwargs,
    _save_final_and_sync_tb,
    _select_handoff_checkpoint,
    _stage_entry_shaping_callbacks,
)

ALGO_CLASS = {"PPO": PPO, "SAC": SAC}


def disconnect_runtime(reason):
    """Flush Drive writes and release the Colab runtime.

    Called whenever training halts — curriculum gate failure or normal
    completion — so an unattended "Run all" never leaves a GPU runtime
    idle. No-op outside Colab or when AUTO_DISCONNECT is False.
    """
    print(f"\n{reason}")
    if not (IN_COLAB and AUTO_DISCONNECT):
        print("Auto-disconnect skipped (not in Colab or AUTO_DISCONNECT is False).")
        return
    if USE_GOOGLE_DRIVE:
        # Flush buffered Drive writes before the runtime dies, otherwise
        # the tail of files written this session (e.g. evaluations.npz)
        # can be lost or truncated.
        from google.colab import drive

        print("Flushing Google Drive writes...")
        drive.flush_and_unmount()
    from google.colab import runtime

    print("Disconnecting runtime in 5 seconds...")
    time.sleep(5)
    runtime.unassign()



# ---------------------------------------------------------------------------
# PPO.load preflight: some runtime images (the known L4 image fault,
# first-runs record §9) SEGFAULT inside PPO.load, so checkpoints written on
# them can never be loaded back — and a segfault kills the kernel and cannot
# be caught by try/except. Prove the load path works on a throwaway model
# BEFORE any training run; the print ahead of the load is flushed first so a
# kernel death is attributable to the load rather than to training.
# ---------------------------------------------------------------------------
import tempfile


class _PreflightEnv(gym.Env):
    """Tiny 2-obs/1-action env; exists only to construct a minimal PPO."""

    observation_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(2,), dtype=np.float32)
    action_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        return np.zeros(2, dtype=np.float32), {}

    def step(self, action):
        return np.zeros(2, dtype=np.float32), 0.0, True, False, {}


_preflight_zip = os.path.join(tempfile.mkdtemp(prefix="ppo_load_preflight_"), "preflight_ppo.zip")
PPO("MlpPolicy", _PreflightEnv(), n_steps=32, batch_size=32, policy_kwargs={"net_arch": [8]}, device="cpu").save(
    _preflight_zip
)
print("PPO.load preflight... (a kernel death HERE means this image cannot load SB3 checkpoints)", flush=True)
try:
    PPO.load(_preflight_zip, device="cpu")
except Exception as _preflight_exc:
    # Mirror the gate-failure cells: release the GPU runtime BEFORE raising,
    # so an unattended "Run all" does not idle a paid runtime on a dead end.
    disconnect_runtime("PPO.load preflight failed: this image cannot load SB3 checkpoints")
    raise RuntimeError(
        "PPO.load preflight failed: this runtime image cannot load SB3 checkpoints "
        "(known L4 image fault); switch runtime or use the fallback runtime version "
        "— do NOT start a training run here."
    ) from _preflight_exc
print("PPO.load preflight passed: checkpoints load back on this runtime.")

try:
    import mediapy

    _HAS_MEDIAPY = True
except ImportError:
    _HAS_MEDIAPY = False
    print("mediapy not installed. Videos will be skipped. Install with: pip install mediapy")

# Chain state (BEHAVIOR_RECIPES_PLAN §4.7). The chain loop fills these as it
# walks BEHAVIOR's chain; the tail cells read them instead of hand-threaded
# per-stage variables.
#   completed_stages: (stage reference, stage_dir) for every node trained or
#     judged in THIS run, in completion order — what the curves cells plot.
#   NODE_RESULTS: stage id -> stage_results dict for every node this run
#     holds results for. A cross-run reused ancestor has none: it writes no
#     CSV row here — its record lives under ancestors/.
#   NODE_HANDOFF: stage id -> the certified handoff its children load from:
#     {"model": <stem>, "vecnorm": <path>, "stage_dir": Path, "run_dir": Path,
#      "run_id": <str | None>, "model_sha256": str, "reused": bool}.
#     run_id is the ancestor's run for a CROSS-run reuse (the child records
#     it as parent_run_id) and None for a node from this run; model_sha256
#     is the digest the next node's reuse rule chains on (D-A17). Certified
#     nodes only — a node that failed its gate never enters it.
completed_stages = []
NODE_RESULTS = {}
NODE_HANDOFF = {}


def chain_results():
    """NODE_RESULTS' values in manifest order — the list every summary and bundle write passes."""
    return [NODE_RESULTS[entry.id] for entry in MANIFEST.stages if entry.id in NODE_RESULTS]


def _eval_forward_vel(model, stage, vecnorm_path, n_episodes=30):
    """Evaluate a trained policy collecting forward velocity, distance, and success rate.

    Thin wrapper around the library's ``eval_policy`` that handles
    environment creation and VecNormalize loading using notebook globals.
    """
    _algo_kwargs = STAGE_CONFIGS[stage].get(f"{ALGORITHM.lower()}_kwargs", {})
    eval_env = create_vec_env(
        SPECIES_CFG,
        STAGE_CONFIGS,
        stage,
        1,
        EVALUATION_SEED,
        algorithm=ALGORITHM.lower(),
        gamma=_algo_kwargs.get("gamma"),
        plant_identity=PLANT_IDENTITY,
    )
    if vecnorm_path and Path(vecnorm_path).exists():
        load_vecnorm_stats(vecnorm_path, eval_env, current_plant=PLANT_IDENTITY)
    eval_env.training = False
    eval_env.norm_reward = False

    rewards, lengths, fwd_vels, successes, distances = eval_policy(
        model,
        eval_env,
        SPECIES_CFG.success_keys,
        n_episodes=n_episodes,
    )
    eval_env.close()
    return rewards, lengths, fwd_vels, successes, distances


def display_stage_videos(stage, stage_dir):
    """Display saved stage videos inline using mediapy.

    Finds all ``.mp4`` files in *stage_dir* and plays them inline.
    This is used after ``generate_stage_artifacts`` has already recorded
    the videos to disk.
    """
    if not _HAS_MEDIAPY:
        print(f"Skipping video display for stage {stage} (mediapy not installed).")
        return
    from environments.shared.reporting import stage_layout

    stage_dir = Path(stage_dir)
    # Resolved through stage_layout: replays live in `replays/` now, and this
    # also finds them in a legacy flat stage directory.
    mp4s = sorted(p for p in stage_layout.iter_replay_files(stage_dir) if p.suffix == ".mp4")
    if not mp4s:
        print(f"No videos found in {stage_dir}")
        return
    for mp4 in mp4s:
        print(f"Playing: {mp4.name}")
        mediapy.show_video(mediapy.read_video(str(mp4)), fps=50)


def train_stage(
    stage,
    timesteps,
    *,
    load_path=None,
    run_dir=None,
    vecnorm_path=None,
    task_load_mode,
    parent_run_id=None,
    label=None,
    eval_freq=50_000,
    save_freq=100_000,
):
    """Train one node of the behavior chain.

    Delegates algorithm setup, model creation, callback construction,
    and checkpoint saving to shared helpers in ``train_base.py``.
    Notebook-specific concerns (interactive printing, the stage record,
    evaluation) are handled here; the gate is judged afterwards by
    ``generate_stage_artifacts`` and enforced by the chain loop.

    ``task_load_mode`` is REQUIRED and never inferred. The chain loop passes
    ``"initialize_next_stage"`` when the node loads its parent's handoff
    (its declared edge, recorded as lineage) and ``"resume_same_stage"`` for
    a root that loads nothing; the RESUME cell passes ``"resume_same_stage"``
    for a load of this node's OWN periodic checkpoint. A checkpoint and its
    VecNormalize sidecar always travel together. ``parent_run_id`` names
    the run a reused certified ancestor came from (``None`` for a parent
    trained in this run); ``label`` is the free-text run label recorded
    beside the hyperparameter digest (decision D-A21).

    Returns (model, best_model_path, final_model_path, stage_dir, vecnorm_save_path, stage_results).
    """
    stage_start = time.time()

    config = STAGE_CONFIGS[stage]
    # Semantic stage support (stage manifest): integer stages keep their
    # historical stage{N} labels; a semantic stage such as "recovery" labels
    # its artifacts by id. Warm-up keys on the node's declared EDGE
    # (warm_start_from), never on its position (BEHAVIOR_RECIPES_PLAN §4.2).
    NODE = MANIFEST.resolve(stage)
    if task_load_mode not in ("initialize_next_stage", "resume_same_stage"):
        raise ValueError(
            "task_load_mode must be declared as 'initialize_next_stage' (the parent's handoff) or "
            f"'resume_same_stage' (this node's own checkpoint, or nothing), not {task_load_mode!r}"
        )
    if bool(load_path) != bool(vecnorm_path):
        raise ValueError(
            "load_path and vecnorm_path must be given together: a checkpoint is only ever loaded under "
            "the VecNormalize statistics it was trained with"
        )
    if task_load_mode == "initialize_next_stage" and not load_path:
        raise ValueError(f"{NODE.id!r}: 'initialize_next_stage' declares a load from the parent's handoff, but no load_path was given")
    if task_load_mode == "initialize_next_stage" and NODE.warm_start_from is None:
        raise ValueError(
            f"{NODE.id!r} is a root of the {SPECIES} manifest (no warm_start_from), so it cannot enter from a "
            "parent under 'initialize_next_stage'; a root trains from scratch or resumes its own checkpoint"
        )
    entering_from_parent = task_load_mode == "initialize_next_stage"
    task_fingerprint = derive_stage_task_fingerprint(
        species=SPECIES,
        stage=stage,
        backend="stable-baselines3",
        env_kwargs=config.get("env_kwargs", {}),
        plant_identity=PLANT_IDENTITY.to_dict(),
    )
    sb3 = _ensure_sb3()

    # Directories — each node gets a subdirectory within the run directory
    if run_dir is None:
        run_dir = repo_root / "logs"
    # Directory names sort in curriculum order and carry the id (01_stance,
    # 02_recovery, ...); files inside keep their stage_label prefixes.
    stage_dir = Path(run_dir) / stage_dirname(SPECIES, stage)
    # D-A20: a directory that already records a stage (stage_config.json or
    # gate_verdict.json) is never overwritten unless this is an explicit
    # same-stage resume of it. The mode passed is the EFFECTIVE one — the one
    # save_stage_config records below (a root that loads nothing loads nothing).
    refuse_occupied_stage_dir(stage_dir, task_load_mode=task_load_mode if load_path else None)
    stage_dir.mkdir(parents=True, exist_ok=True)
    model_dir = stage_dir / "models"
    model_dir.mkdir(exist_ok=True)
    # D-A15: a same-stage resume continues a recorded stage, so its duration
    # accumulates on the record's — read BEFORE the config re-save below
    # rewrites the run block. (That re-save keeps the initialize_next_stage
    # edge the node entered on and records the periodic checkpoint under
    # config.RESUME_LINEAGE_KEYS, so a resumed-then-judged node still chains
    # by digest for reuse — ancestors rule 4.)
    prior_duration = read_stage_duration(stage_dir) if (load_path and task_load_mode == "resume_same_stage") else None

    # Resolve stage transition parameters up front so the exact values
    # (whether from TOML or defaults) are recorded in the saved config.
    cur_kwargs = config.get("curriculum_kwargs", {})
    extra_meta = {"seed": SEED, "n_envs": N_ENVS, "timesteps": timesteps}
    if entering_from_parent:
        warmup_timesteps = cur_kwargs.get("warmup_timesteps", 100_000)
        warmup_clip_range = cur_kwargs.get("warmup_clip_range", 0.02)
        warmup_ent_coef = cur_kwargs.get("warmup_ent_coef", 0.02)
        ramp_timesteps = cur_kwargs.get("ramp_timesteps", 500_000)
        ramp_start_value = cur_kwargs.get("ramp_start_value", 0.1)
        target_fwd_weight = config["env_kwargs"].get("forward_vel_weight", 1.0)
        extra_meta.update(
            {
                "warmup_timesteps": warmup_timesteps,
                "warmup_clip_range": warmup_clip_range,
                "warmup_ent_coef": warmup_ent_coef,
                "ramp_timesteps": ramp_timesteps,
                "ramp_start_value": ramp_start_value,
                "forward_vel_weight": target_fwd_weight,
            }
        )

    print(f"{'=' * 60}")
    print(f"Node {NODE.id} (stage {stage}): {config['name']} ({ALGORITHM})")
    print(f"Description: {config['description']}")
    print(f"Timesteps: {timesteps:,}")
    print(f"Log dir: {stage_dir}")
    if vecnorm_path:
        print(f"VecNormalize loaded from: {vecnorm_path}")
    print(f"{'=' * 60}")

    if entering_from_parent:
        # The same refusal train() makes (BEHAVIOR_RECIPES_PLAN §4.2 retarget
        # 3): a boundary-crossing load must come from THIS node's declared
        # parent; a checkpoint with no fingerprint warns through the dated
        # valve. Checked before the stage record is written.
        _parent_entry = MANIFEST.parent_of(stage)
        validate_declared_parent(
            read_checkpoint_task_fingerprint(load_path),
            declared_parent=_parent_entry.reference if _parent_entry is not None else None,
            species=SPECIES,
            child_stage=NODE.reference,
            artifact=str(load_path),
        )

    # Save reward weights, hyperparameters, and resolved stage transition
    # params for reproducibility.
    cfg_path = save_stage_config(
        stage_dir,
        stage,
        config,
        ALGORITHM,
        extra=extra_meta,
        env_class=EnvClass,
        species=SPECIES,
        plant_identity=PLANT_IDENTITY,
        task_fingerprint=task_fingerprint,
        load_path=load_path,
        load_mode=task_load_mode if load_path else None,
        parent_run_id=parent_run_id,
        label=label,
    )
    print(f"Stage config saved to: {cfg_path}")

    # Create environments using library infrastructure.
    use_subproc = ALGORITHM.lower() == "sac" and N_ENVS > 1
    if use_subproc:
        print(f"Using SubprocVecEnv for SAC ({N_ENVS} parallel workers)")
    _algo_kwargs_cfg = config.get(f"{ALGORITHM.lower()}_kwargs", {})
    _alg_gamma = _algo_kwargs_cfg.get("gamma")
    train_env = create_vec_env(
        SPECIES_CFG,
        STAGE_CONFIGS,
        stage,
        N_ENVS,
        SEED,
        use_subproc=use_subproc,
        algorithm=ALGORITHM.lower(),
        gamma=_alg_gamma,
        plant_identity=PLANT_IDENTITY,
    )
    eval_env = create_vec_env(
        SPECIES_CFG,
        STAGE_CONFIGS,
        stage,
        1,
        CHECKPOINT_SELECTION_SEED,
        algorithm=ALGORITHM.lower(),
        gamma=_alg_gamma,
        plant_identity=PLANT_IDENTITY,
    )
    # Continuation semantics apply only to a periodic checkpoint of THIS
    # stage under resume_same_stage (and PPO — SAC's replay buffer is not
    # persisted); curated checkpoints keep the fresh-counter behavior.
    # Mirrors train_base.train().
    resuming = _is_resume_continuation(
        load_path,
        task_load_mode=task_load_mode,
        stage_lbl=stage_label(stage),
        algorithm=ALGORITHM.lower(),
    )

    # Load VecNormalize stats from prior stage (or this stage's own periodic
    # sidecar on a same-stage resume, where the return-normalization
    # statistics are carried forward too: the reward distribution is
    # unchanged, so resetting ret_rms only distorts the first post-resume
    # updates — mirrors train_base._load_vecnorm_into_envs, review TC6).
    if vecnorm_path:
        if not load_vecnorm_stats(
            vecnorm_path,
            train_env,
            eval_env,
            current_plant=PLANT_IDENTITY,
            carry_ret_rms=(task_load_mode == "resume_same_stage"),
            # Invariant 8 (BEHAVIOR_RECIPES_PLAN §4.6): under a command mode the
            # loaded statistics' command slice is reseeded to mean 0 / var 1 on
            # both destinations — EXCEPT on a same-stage resume, whose sidecar
            # already holds the statistics the policy trained under (mirrors
            # train_base._load_vecnorm_into_envs). Always False in Phase C,
            # where only command_mode = "none" exists.
            reseed_command_slice=(
                config.get("env_kwargs", {}).get("command_mode", "none") != "none"
                and task_load_mode != "resume_same_stage"
            ),
        ):
            # Fail closed, mirroring train_base._load_vecnorm_into_envs
            # (review TC5): training the loaded policy under fresh
            # mean-0/var-1 statistics feeds it wildly mis-scaled
            # observations and collapses it within the first updates.
            raise FileNotFoundError(
                f"VecNormalize sidecar not found: {vecnorm_path}. Refusing to train the "
                "loaded checkpoint under fresh normalization statistics."
            )
    else:
        eval_env.training = False
        eval_env.norm_reward = False

    # Build algorithm kwargs using shared helper (handles LR schedules,
    # clip range annealing, and TensorBoard setup).
    alg_kwargs, local_tb_dir, gcs_tb_path = _prepare_alg_kwargs(
        config,
        ALGORITHM.lower(),
        VERBOSE,
        stage_dir,
        use_tensorboard=True,
    )

    # Create or load model using shared helper
    if load_path:
        print(f"Loading model from: {load_path}")
    model = _create_or_load_model(
        sb3,
        ALGORITHM.lower(),
        alg_kwargs,
        train_env,
        load_path,
        plant_identity=PLANT_IDENTITY,
        task_fingerprint=task_fingerprint,
        task_load_mode=task_load_mode,
    )

    # A same-stage resume continues the interrupted run: the loaded step
    # counter is preserved (reset_num_timesteps=False in model.learn below),
    # so `timesteps` means "this many MORE steps" and every step-anchored
    # mechanism (checkpoint numbering, retention, schedule progress, entropy
    # decay, eval history timesteps) stays on the run's cumulative axis
    # (mirrors train_base.train()).
    loaded_steps = int(getattr(model, "num_timesteps", 0)) if resuming else 0
    target_timesteps = loaded_steps + timesteps
    if resuming and loaded_steps > 0:
        print(
            f"Resuming at {loaded_steps:,} cumulative steps; "
            f"training {timesteps:,} more (target {target_timesteps:,})."
        )

    # Build callbacks using shared helper (includes EvalCallback,
    # CheckpointCallback, DiagnosticsCallback, and EvalCollapseEarlyStopCallback).
    callbacks, eval_callback, _save_vecnorm_cb = _build_core_callbacks(
        sb3,
        eval_env,
        model_dir,
        stage_dir,
        stage,
        N_ENVS,
        eval_freq=eval_freq,
        # 100k cadence (was 500k): a Colab session cap now discards at most
        # ~30 min of training instead of ~2.2 h; retention keeps 5
        # step-points, so storage is unchanged (review CO4).
        save_freq=save_freq,
        verbose=VERBOSE,
        stage_config=config,
        species=SPECIES,
        # The cumulative target this session trains to, not the TOML budget:
        # the budget-anchored advisories must follow resumes (review TC9).
        total_timesteps=target_timesteps,
    )
    if resuming:
        from environments.shared.curriculum import seed_resume_eval_state

        # Seed best-model trackers and the in-memory eval history from the
        # published record (falling back to the interrupted stage directory
        # the checkpoint came from), so the first post-resume eval cannot
        # overwrite a better pre-interruption best_model / robust_best_model,
        # and evaluations.npz keeps the whole run. Entries newer than the
        # resumed checkpoint are dropped so timesteps stay monotonic.
        seed_candidates = [
            stage_dir / "evaluations.npz",
            Path(load_path).resolve().parent.parent / "evaluations.npz",
        ]
        seed_path = next((c for c in seed_candidates if c.exists()), seed_candidates[0])
        seed_resume_eval_state(eval_callback, callbacks, seed_path, max_timesteps=loaded_steps)

    # Optional PPO entropy-coefficient decay (set ent_coef_end in the TOML).
    # The decay anchor is an ABSOLUTE step count: on a resume the default
    # (when the TOML sets no ent_coef_decay_timesteps) must be the cumulative
    # target, not this call's remaining budget — otherwise a late resume
    # decays over a compressed horizon (mirrors train_base.train()).
    ent_decay_cb = _maybe_ent_coef_decay_callback(config, ALGORITHM.lower(), target_timesteps)
    if ent_decay_cb is not None:
        callbacks.append(ent_decay_cb)

    # Stage-ENTRY shaping only (task_load_mode "initialize_next_stage"):
    # warm up the value function and ramp the forward velocity reward to
    # prevent catastrophic forgetting of balance behaviours. A
    # resume_same_stage load just passed an exact task-fingerprint identity
    # check and must resume the exact task, so it gets neither (review F4).
    # Built by the shared helper, never inline: it carries the
    # forward_vel_weight > 0 ramp guard (review F5), and the inline copy this
    # cell used to keep is where the 20260821 recovery behavior lost it.
    shaping = _stage_entry_shaping_callbacks(
        config,
        task_load_mode=task_load_mode,
        parent_id=NODE.warm_start_from,
        load_path=load_path,
    )
    # Unfiltered: SAC gets the same stage-entry warm-up the CLI gives it
    # (gap-review DU1, closed by BEHAVIOR_RECIPES_PLAN §4.7).
    callbacks.extend(shaping)

    # Train
    model.learn(
        total_timesteps=timesteps,
        callback=sb3["CallbackList"](callbacks),
        progress_bar=VERBOSE >= 1,
        # On resume, keep the checkpoint's cumulative counter: SB3 then
        # trains `timesteps` MORE steps, checkpoint filenames stay
        # cumulative (so retention keeps the newest, not the stale
        # pre-crash set), and progress-based schedules continue from
        # where the interrupted run left off.
        reset_num_timesteps=not resuming,
    )

    # Actual CUMULATIVE steps trained — differs from the target when a
    # callback (e.g. EvalCollapseEarlyStopCallback) ended training early.
    # On a resume this includes the loaded checkpoint's steps.
    actual_timesteps = int(model.num_timesteps)
    if actual_timesteps < target_timesteps:
        print(f"\nNOTE: training stopped early at {actual_timesteps:,} of {target_timesteps:,} timesteps.")

    # Save final model and sync TensorBoard using shared helper
    final_path = _save_final_and_sync_tb(
        model,
        train_env,
        model_dir,
        stage,
        local_tb_dir,
        gcs_tb_path,
    )
    final_vecnorm_path = str(final_path) + "_vecnorm.pkl"
    print(f"\nFinal model saved to: {final_path}.zip")
    print(f"VecNormalize stats saved to: {final_vecnorm_path}")

    # D-A15: the stage record carries the real training duration — on a
    # resume, the interrupted session's plus this one's — so a node judged
    # later (the chain loop's JUDGE branch) reports it honestly.
    stage_duration = (prior_duration or 0.0) + (time.time() - stage_start)
    record_stage_duration(stage_dir, stage_duration)

    train_env.close()
    eval_env.close()

    _model, _handoff_stem, _final_model_path, _handoff_vecnorm, _stage_results = evaluate_stage_checkpoints(
        stage,
        stage_dir,
        final_path=final_path,
        final_vecnorm_path=final_vecnorm_path,
        timesteps=actual_timesteps,
        duration_seconds=stage_duration,
        model=model,
    )
    return _model, _handoff_stem, _final_model_path, stage_dir, _handoff_vecnorm, _stage_results


def evaluate_stage_checkpoints(
    stage,
    stage_dir,
    *,
    final_path,
    final_vecnorm_path,
    timesteps,
    duration_seconds,
    model=None,
):
    """Evaluate a trained node's final and selected checkpoints and build its stage_results.

    The tail of ``train_stage``, split out so the chain loop's JUDGE branch
    can run it over a node whose budget was spent by the RESUME cell but
    whose gate was never judged (no ``gate_verdict.json``). ``model`` is the
    in-memory final model right after training; ``None`` loads
    ``<final_path>.zip`` back, validates its plant and takes its cumulative
    step counter as ``timesteps``. Writes the final and selected evidence
    CSVs, each bound to the checkpoint and VecNormalize sidecar it rolled
    under, and selects the handoff checkpoint through the shared selector.

    Returns (model, handoff_stem, final_model_path, handoff_vecnorm_path, stage_results).
    """
    config = STAGE_CONFIGS[stage]
    stage_dir = Path(stage_dir)
    model_dir = stage_dir / "models"
    AlgoClass = ALGO_CLASS[ALGORITHM.upper()]
    final_path = Path(final_path)
    final_vecnorm_path = str(final_vecnorm_path)
    vecnorm_save_path = final_vecnorm_path
    if model is None:
        model = AlgoClass.load(str(final_path))
        validate_model_plant(model, PLANT_IDENTITY, artifact=f"{final_path}.zip")
        timesteps = int(getattr(model, "num_timesteps", 0)) or timesteps

    # Evaluate the final model with its matching VecNormalize stats.
    episode_rewards, episode_lengths, episode_fwd_vels, episode_successes, episode_distances = _eval_forward_vel(
        model,
        stage,
        final_vecnorm_path,
        n_episodes=30,
    )
    final_eval_path = _lib_save_evaluation_episodes(
        stage_dir,
        rewards=episode_rewards,
        lengths=episode_lengths,
        forward_velocities=episode_fwd_vels,
        distances=episode_distances,
        successes=episode_successes,
        evaluation_seed=EVALUATION_SEED,
        checkpoint_label="final",
        checkpoint_path=f"{final_path}.zip",
        normalization_path=final_vecnorm_path,
    )
    print(f"Final-model episode evidence saved to: {final_eval_path}")
    mean_reward = float(np.mean(episode_rewards))
    std_reward = float(np.std(episode_rewards))
    mean_length = float(np.mean(episode_lengths))
    std_length = float(np.std(episode_lengths))
    mean_fwd_vel = float(np.mean(episode_fwd_vels))
    std_fwd_vel = float(np.std(episode_fwd_vels))
    mean_distance = float(np.mean(episode_distances))
    # The training envs are closed by now: a bare env of this node's task
    # answers for its control timestep.
    _probe_env = EnvClass(**config.get("env_kwargs", {}))
    sim_dt = float(_probe_env.dt)
    _probe_env.close()
    mean_success_rate = float(np.mean(episode_successes))
    print(f"Eval (final model): mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")
    print(f"Eval: mean_length={mean_length:.1f} +/- {std_length:.1f} steps ({mean_length * sim_dt:.2f}s sim time)")
    print(f"Eval: mean_forward_vel={mean_fwd_vel:.2f} +/- {std_fwd_vel:.2f} m/s")
    print(f"Eval: mean_distance_traveled={mean_distance:.2f} m")
    print(f"Eval: success_rate={mean_success_rate:.0%}")

    # Build base results dict from on-disk eval data (evaluations.npz),
    # then enrich with the live 30-episode evaluation metrics above.
    stage_results = build_stage_results_from_eval_data(
        stage_dir,
        stage,
        config,
        timesteps=timesteps,
        duration_seconds=duration_seconds,
    )
    best_eval_reward = stage_results["best_eval_reward"]
    best_eval_std = stage_results["best_eval_std"]
    best_eval_timestep = stage_results["best_eval_timestep"]
    if best_eval_reward != "":
        print(f"Best model eval:  mean_reward={best_eval_reward} +/- {best_eval_std} (at {best_eval_timestep:,} steps)")

    # Override with richer live-eval metrics
    stage_results.update(
        {
            "mean_reward": mean_reward,
            "std_reward": std_reward,
            "mean_episode_length": mean_length,
            "std_episode_length": std_length,
            "mean_forward_vel": mean_fwd_vel,
            "std_forward_vel": std_fwd_vel,
            "mean_distance_traveled": mean_distance,
            "mean_success_rate": mean_success_rate,
            "sim_dt": sim_dt,
        }
    )

    # The SELECTED checkpoint: next-stage loading, the evidence CSV below, the
    # replay video, and the stance gate report must all describe the same
    # policy. `_select_handoff_checkpoint` is the one selector they share --
    # this cell used to carry a private copy of the preference order, which is
    # how the replay ended up showing `best_model` while
    # `evaluation_selected.csv` was evidence for `robust_best_model`.
    #
    # It prefers the risk-adjusted robust_best_model (highest mean - std eval)
    # over SB3's mean-reward best_model: a high mean can be propped up by a good
    # mode while a fat failure tail is already growing (run 20260709_185946).
    # It requires the matched VecNormalize stats, so it cannot return a hybrid.
    _handoff = _select_handoff_checkpoint(model_dir)
    if _handoff is None:
        best_model_zip = model_dir / "best_model.zip"
        best_vecnorm_candidate = str(model_dir / "best_model_vecnorm.pkl")
        if best_model_zip.exists():
            raise FileNotFoundError(
                f"Selected checkpoint {best_model_zip} is missing its matched VecNormalize state "
                f"{best_vecnorm_candidate}; refusing to evaluate or export a hybrid checkpoint."
            )
    else:
        _selected_name, _selected_path, best_vecnorm_candidate = _handoff
        best_model_zip = model_dir / f"{_selected_name}.zip"
        print(f"Selected checkpoint: {_selected_name}")
    best_model_reward, best_model_std_reward = "", ""
    best_model_length, best_model_std_length = "", ""
    best_model_fwd_vel, best_model_std_fwd_vel = "", ""
    best_model_distance = ""
    best_model_success_rate = ""
    if best_model_zip.exists():
        best_path = model_dir / best_model_zip.stem
        model = AlgoClass.load(str(best_path))
        validate_model_plant(model, PLANT_IDENTITY, artifact=str(best_model_zip))
        output_path = str(best_path)
        print(f"Loaded best model for next-stage: {best_path}.zip")

        # Guaranteed by _select_handoff_checkpoint, which only returns a
        # candidate whose matched statistics exist. Kept as an assertion
        # because exporting a hybrid checkpoint is silent and unrecoverable.
        if not Path(best_vecnorm_candidate).exists():
            raise FileNotFoundError(
                f"Selected checkpoint {best_model_zip} is missing its matched VecNormalize state "
                f"{best_vecnorm_candidate}; refusing to evaluate or export a hybrid checkpoint."
            )
        vecnorm_save_path = best_vecnorm_candidate
        print(f"Using matched VecNormalize for selected checkpoint: {vecnorm_save_path}")

        # Evaluate the best model over 30 episodes
        print("Evaluating best model (30 episodes)...")
        bm_rewards, bm_lengths, bm_fwd_vels, bm_successes, bm_distances = _eval_forward_vel(
            model,
            stage,
            vecnorm_save_path,
            n_episodes=30,
        )
        selected_eval_path = _lib_save_evaluation_episodes(
            stage_dir,
            rewards=bm_rewards,
            lengths=bm_lengths,
            forward_velocities=bm_fwd_vels,
            distances=bm_distances,
            successes=bm_successes,
            evaluation_seed=EVALUATION_SEED,
            checkpoint_label="selected",
            checkpoint_path=best_model_zip,
            normalization_path=vecnorm_save_path,
        )
        print(f"Selected-model episode evidence saved to: {selected_eval_path}")
        best_model_reward = round(float(np.mean(bm_rewards)), 2)
        best_model_std_reward = round(float(np.std(bm_rewards)), 2)
        best_model_length = round(float(np.mean(bm_lengths)), 1)
        best_model_std_length = round(float(np.std(bm_lengths)), 1)
        best_model_fwd_vel = round(float(np.mean(bm_fwd_vels)), 2)
        best_model_std_fwd_vel = round(float(np.std(bm_fwd_vels)), 2)
        best_model_distance = round(float(np.mean(bm_distances)), 2)
        best_model_success_rate = round(float(np.mean(bm_successes)), 2)
        print(f"Best model eval:  mean_reward={best_model_reward} +/- {best_model_std_reward}")
        print(f"Best model eval:  mean_length={best_model_length} +/- {best_model_std_length}")
        print(f"Best model eval:  mean_fwd_vel={best_model_fwd_vel} +/- {best_model_std_fwd_vel} m/s")
        print(f"Best model eval:  mean_distance={best_model_distance} m")
        print(f"Best model eval:  success_rate={best_model_success_rate:.0%}")
    else:
        output_path = str(final_path)
        selected_eval_path = _lib_save_evaluation_episodes(
            stage_dir,
            rewards=episode_rewards,
            lengths=episode_lengths,
            forward_velocities=episode_fwd_vels,
            distances=episode_distances,
            successes=episode_successes,
            evaluation_seed=EVALUATION_SEED,
            checkpoint_label="selected",
            checkpoint_path=f"{final_path}.zip",
            normalization_path=final_vecnorm_path,
        )
        print(f"Selected-model episode evidence saved to: {selected_eval_path}")
        best_model_reward = round(float(np.mean(episode_rewards)), 2)
        best_model_std_reward = round(float(np.std(episode_rewards)), 2)
        best_model_length = round(float(np.mean(episode_lengths)), 1)
        best_model_std_length = round(float(np.std(episode_lengths)), 1)
        best_model_fwd_vel = round(float(np.mean(episode_fwd_vels)), 3)
        best_model_std_fwd_vel = round(float(np.std(episode_fwd_vels)), 3)
        best_model_distance = round(float(np.mean(episode_distances)), 3)
        best_model_success_rate = round(float(np.mean(episode_successes)), 4)

    # The curriculum gate is NOT evaluated here. `generate_stage_artifacts`
    # evaluates it, through the one shared `reporting.gates.evaluate_stage_gate`,
    # and records `gate_passed` / `publication_gate_passed` / `gate_failures`
    # onto this dict; the chain loop enforces it.
    #
    # This cell used to carry its own checklist over min_avg_reward /
    # min_avg_episode_length / min_avg_forward_vel / min_success_rate. It knew
    # nothing about `gate_kind`, so when Tyrannosaurus Rex stage 1 moved to
    # stance_quality/v1 and retired min_avg_episode_length, the checklist
    # quietly degraded to a reward comparison alone -- which the zero-action
    # statue clears by 68%. Run 20260802_203215 recorded
    # publication_gate_passed = True beside a stance_gate_report.txt reading
    # GATE: FAIL at 10.6x the duty ceiling, and advanced to stage 2 on it.
    # Keep the rule in the library where both backends and the sweep worker
    # read the same copy.

    # Update stage_results with model paths and best-model eval
    stage_results.update(
        {
            "model_path": output_path,
            "final_model_path": str(final_path),
            "vecnorm_path": vecnorm_save_path,
            "final_vecnorm_path": final_vecnorm_path,
            "best_model_reward": best_model_reward,
            "best_model_std_reward": best_model_std_reward,
            "best_model_length": best_model_length,
            "best_model_std_length": best_model_std_length,
            "best_model_fwd_vel": best_model_fwd_vel,
            "best_model_std_fwd_vel": best_model_std_fwd_vel,
            "best_model_distance": best_model_distance,
            "best_model_success_rate": best_model_success_rate,
        }
    )

    return model, output_path, str(final_path), vecnorm_save_path, stage_results


def write_training_summary(run_dir, stage_results_list, species=None):
    """Write a training summary text file to the run directory."""
    if species is None:
        species = SPECIES
    path = _lib_write_training_summary(
        run_dir,
        stage_results_list,
        species=species,
        algorithm=ALGORITHM,
        seed=SEED,
        n_envs=N_ENVS,
        quick_test=QUICK_TEST,
    )
    summary_text = path.read_text()
    print(f"\nTraining summary saved to: {path}")
    print(summary_text)


def save_run_bundle(stage_results_list, run_dir=None, species=SPECIES):
    """Regenerate the canonical, Drive-portable bundle for the nodes this run holds results for.

    The bundle is `complete` only when BEHAVIOR's node (the target
    deliverable) is present and certified and every deliverable present is
    certified; a chain stopped above the target leaves it `partial`, which
    still publishes every certified deliverable (result schema v4).

    Seed replication (BEHAVIOR_RECIPES_PLAN §4.5, decision D-B16): the sibling
    runs under this species/algorithm log directory that certified the same
    recipe (same task, plant, gate and hyperparameter digests) on another seed
    are recorded as each deliverable's replicates, and a deliverable with fewer
    certifying runs than its config's `certification_seeds` is labelled
    provisional. Re-run this cell once a replicate finishes to count it: a
    partial bundle is rebuilt, and a complete one regenerates its derived
    artifacts when the replication record is its only change (everything
    else in it stays immutable).
    """
    if run_dir is None:
        run_dir = RUN_DIR
    paths = _lib_save_result_bundle(
        stage_results_list,
        target_deliverable=TARGET_NODE.key,
        stage_configs=STAGE_CONFIGS,
        species=species,
        algorithm=ALGORITHM,
        seed=SEED,
        run_dir=run_dir,
        backend="stable-baselines3",
        hardware=HARDWARE_LABEL,
        parallel_envs=N_ENVS,
        evaluation_episodes=30,
        evaluation_seeds=[CHECKPOINT_SELECTION_SEED, EVALUATION_SEED],
        seed_roles={
            "training": SEED,
            "checkpoint_selection_evaluation": CHECKPOINT_SELECTION_SEED,
            "publication_evaluation": EVALUATION_SEED,
            "certification_panel": PUBLICATION_SEED_START,
        },
        plant_identity=PLANT_IDENTITY.to_dict(),
        run_id=RUN_ID,
        repository_root=repo_root,
        replicates=discover_replicates_for_run(run_dir, species=species, plant_identity=PLANT_IDENTITY),
    )
    print("\nResult bundle updated:")
    for name, path in paths.items():
        print(f"  {name}: {path}")
    return paths



if not globals().get("COMMAND_TERRAIN_BEHAVIOR", False):
    print(f"Training infrastructure ready. Algorithm: {ALGORITHM}")
    print("DiagnosticsCallback enabled: per-component rewards, obs/action stats,")
    print("VecNormalize tracking, and termination reasons")
    print("will log to TensorBoard.")
else:
    print("Behavior runner ready: PPO progress CSV, episode manifests, and diagnostic evaluation.")

### Visualization Functions

In [ ]:
from environments.shared.visualization import (
    plot_diagnostics_graphs as _lib_plot_diagnostics_graphs,
)
from environments.shared.visualization import (
    plot_training_curves as _lib_plot_training_curves,
)


def plot_training_curves(stage_dirs, stage_configs, algo_name, save_path=None):
    """Plot evaluation reward, episode length, tilt angle, and forward velocity curves.

    Thin wrapper around the shared library function that fills in the
    notebook's SPECIES global and calls ``plt.show()`` for inline display.
    """
    _lib_plot_training_curves(
        stage_dirs,
        stage_configs,
        species=SPECIES,
        algorithm=algo_name,
        save_path=save_path,
    )
    if save_path is not None:
        print(f"Training curves saved to: {save_path}")
    plt.show()


def plot_diagnostics_graphs(stage_dirs, stage_configs, algo_name, save_dir=None, _show=True):
    """Create diagnostic figures for locomotion health and behavioral metrics.

    Thin wrapper around the shared library function that fills in the
    notebook's SPECIES global.  When ``_show`` is True (default), figures
    are displayed inline; otherwise they are closed after saving.
    """
    fig1, fig2 = _lib_plot_diagnostics_graphs(
        stage_dirs,
        stage_configs,
        species=SPECIES,
        algorithm=algo_name,
        save_dir=save_dir,
        show=_show,
    )
    if _show:
        plt.show()


print("Visualization functions ready (shared library).")

## 4b. Run the selected direction or terrain behavior

This cell uses the same runner as the command line. It saves matched periodic checkpoints directly to the chosen Drive/local directory. Existing output directories are never overwritten. Canonical `stand`, `walk` and `hunt` selections skip this cell.


In [ ]:
# Direction and terrain training; the canonical curriculum uses the existing cells below.
# ===== RUN DIRECTION OR TERRAIN BEHAVIOR =====
if COMMAND_TERRAIN_BEHAVIOR:
    from environments.shared.behavior_notebook import run_notebook_behavior

    BEHAVIOR_RESULT = run_notebook_behavior(BEHAVIOR_PLAN)
    print(f"Behavior session {BEHAVIOR_RESULT['status']}; artifacts saved to {BEHAVIOR_PLAN.output_dir}")


## 5. Train the behavior chain

One loop walks `BEHAVIOR`'s chain from the species' stage manifest, root first, and for each node does exactly one of:

1. **Reuse** a certified checkpoint — from this run (a re-run of this cell in the same `RUN_DIR`) or, for an ancestor, from `TRUNK_FROM` — when the reuse rule holds: a passed gate verdict, the same task digest and plant, and a checkpoint that descends from the parent resolved here. A cross-run ancestor is recorded under `ancestors/` and loaded from where it lives, never copied; the target node is never reused across runs.
2. **Judge** a node that was trained but never gated (its final checkpoint exists, `gate_verdict.json` does not — the RESUME cell below spent its budget): evaluate its checkpoints and judge the gate.
3. **Train** it otherwise, warm-started from its parent's handoff checkpoint and VecNormalize sidecar along the declared edge (`initialize_next_stage`); a root loads nothing. For a frozen-null gate kind (recovery) the gate's thresholds and null panels freeze **before** training and the policy panel rolls after, on the same frozen seeds.

Every trained or judged node writes its artifacts, the training summary and the run bundle **before** its verdict is enforced, so a failed gate still publishes every certified deliverable above it; the runtime is then released and the loop raises. `RETRAIN_FROM` trains the named node and everything below it instead of reusing them; a run directory that already records a node is refused, so a new variant is a new `RUN_ID`. Under `BEHAVIOR = "stand"` on a species with a recovery stage, the recovery verdict is enforced like any other node's.

In [ ]:
# ===== BEHAVIOR CHAIN LOOP =====
# Walks BEHAVIOR's chain root-first (BEHAVIOR_RECIPES_PLAN §4.7, decision D5)
# and, per node, does exactly one of:
#   (1) REUSE — a certified checkpoint already exists in RUN_DIR (a same-run
#       re-run) or, for an ancestor, in TRUNK_FROM's run (§4.2: passed verdict,
#       same task digest and plant, chained by digest onto the parent resolved
#       here — D-A17). Cross-run ancestors are recorded under ancestors/, never
#       copied; the target node is never reused across runs (D-A18).
#   (3) JUDGE — trained (final checkpoint + sidecar exist; the RESUME cell spent
#       its budget) but never judged: evaluate, then gate it like a trained node.
#   (2) TRAIN — otherwise, from the parent's handoff along the declared edge
#       ("initialize_next_stage"); a root loads nothing.
# A frozen-null gate kind (recovery) freezes its resolution BEFORE the node
# trains and rolls the policy panel after. Every trained or judged node writes
# its artifacts and the run bundle BEFORE the verdict is enforced — a failed
# gate still publishes every certified deliverable above it — then releases
# the runtime and raises. RETRAIN_FROM (D-A19) removes reuse for the named node
# and its descendants; a recorded stage directory refuses to be overwritten
# (D-A20), so every variant is a new run.
import json

from environments.shared.ancestors import AncestorReuseError, find_certified_ancestor, record_ancestor
from environments.shared.config import hyperparameter_diff
from environments.shared.curriculum import FROZEN_NULL_GATE_KINDS
from environments.shared.harnesses.freeze_recovery_gate import (
    freeze_recovery_gate,
    roll_policy_panel,
    validate_recovery_resolution,
)
from environments.shared.recovery_evaluation import write_recovery_evidence
from environments.shared.result_bundle import read_gate_verdict, sha256_file
from environments.shared.task_fingerprint import (
    derive_stage_task_fingerprint,  # noqa: F811 - this cell also runs standalone
)

if not globals().get("COMMAND_TERRAIN_BEHAVIOR", False):
    print(f"Behavior {BEHAVIOR!r} -> {TARGET_NODE.id}; chain: {' -> '.join(node.id for node in CHAIN)}")
if globals().get("COMMAND_TERRAIN_BEHAVIOR", False):
    CHAIN = []  # Also clear any stale canonical chain when this cell is rerun alone.
for NODE in CHAIN:
    stage = NODE.reference
    config = STAGE_CONFIGS[stage]
    stage_dir = Path(RUN_DIR) / stage_dirname(SPECIES, stage)
    gate_kind = config.get("curriculum_kwargs", {}).get("gate_kind")
    budget = 50_000 if QUICK_TEST else config["curriculum_kwargs"]["timesteps"]
    print(f"\n{'#' * 60}\n# Node {NODE.id} (stage {stage}): {config['name']} — gate {gate_kind}\n{'#' * 60}")

    # The parent resolves first: a node is satisfied — by a checkpoint from
    # anywhere — only on top of its declared parent's certified handoff.
    parent = MANIFEST.parent_of(NODE.id)
    if parent is not None and parent.id not in NODE_HANDOFF:
        raise RuntimeError(
            f"{NODE.id!r} warm-starts from {parent.id!r}, which is not certified in this session (NODE_HANDOFF). "
            "Re-run this cell from the top of the chain; a parent that failed its gate stops the chain there."
        )
    parent_handoff = NODE_HANDOFF[parent.id] if parent is not None else None
    task_sha256 = derive_stage_task_fingerprint(
        species=SPECIES,
        stage=stage,
        backend="stable-baselines3",
        env_kwargs=config.get("env_kwargs", {}),
        plant_identity=PLANT_IDENTITY.to_dict(),
    )["task_sha256"]

    # D-A19: RETRAIN_FROM covers the named node and every descendant — no reuse
    # candidates at all, they train here. D-A18: the target is only ever reused
    # from THIS run (an earlier run's target is that run's deliverable);
    # ancestors also look in TRUNK_DIR. RUN_DIR is tried first; only the
    # trunk candidate may follow ancestor records (D-A23).
    covered = RETRAIN_NODE is not None and (NODE.id == RETRAIN_NODE.id or RETRAIN_NODE in MANIFEST.ancestors(NODE.id))
    if covered:
        candidates = []
        print(f"Not reusing {NODE.id!r}: RETRAIN_FROM={RETRAIN_FROM!r} covers it. Training it here.")
    elif NODE.id == TARGET_NODE.id:
        candidates = [RUN_DIR]
    else:
        candidates = [RUN_DIR] + ([TRUNK_DIR] if TRUNK_DIR is not None else [])

    # (1) REUSE — every refusal is printed with its reason, never silent.
    ancestor = None
    for candidate in candidates:
        try:
            ancestor = find_certified_ancestor(
                candidate,
                species=SPECIES,
                entry=NODE,
                current_task_sha256=task_sha256,
                plant_identity=PLANT_IDENTITY,
                # D-A22 (rule 7): the block this session would judge the node under —
                # the one save_stage_config records as 'curriculum'.
                current_gate_config=config.get("curriculum_kwargs", {}),
                parent_model_sha256=parent_handoff["model_sha256"] if parent_handoff is not None else None,
                # D-A23: a trunk that itself reused a node resolves it through its
                # ancestors/ record to the run that certified it. Never for RUN_DIR:
                # this run's own record marks a cross-run reuse, not a node of ours.
                follow_records=candidate is not RUN_DIR,
            )
            break
        except AncestorReuseError as exc:
            print(f"Not reusing {NODE.id!r} from {candidate}: {exc}")
    if ancestor is not None:
        same_run = candidate == RUN_DIR
        # D-A21: reuse carries the ancestor's recipe. An edit to this node's
        # algorithm block or shaping keys since the ancestor was trained is
        # IGNORED — say so, and say how to train it here. Never a refusal.
        try:
            _recorded = json.loads((ancestor.stage_dir / "stage_config.json").read_text(encoding="utf-8"))
            ignored_edits = hyperparameter_diff(config, ALGORITHM, _recorded)
        except (OSError, ValueError):
            ignored_edits = ["<unreadable stage_config.json>"]
        if ignored_edits:
            print(
                f"WARNING: reusing certified {NODE.id!r} from run {ancestor.run_id} ignores this run's hyperparameter "
                f"edit: {', '.join(ignored_edits)} differ from the ancestor's recorded stage_config.json, and reuse "
                f"trains nothing. Set RETRAIN_FROM = {NODE.id!r} to train {NODE.id!r} here under the edited configuration."
            )
        if same_run:
            # This run's own certified node (a re-run of this cell in the same
            # RUN_DIR): its results re-enter the bundle from the verdict.
            stage_result = ancestor.verdict.get("stage_result")
            if not isinstance(stage_result, dict):
                raise RuntimeError(f"{ancestor.stage_dir / 'gate_verdict.json'} records no stage_result to re-enter the bundle with")
            NODE_RESULTS[NODE.id] = dict(stage_result)
            completed_stages.append((stage, ancestor.stage_dir))
            print(f"Reusing this run's certified {NODE.id!r}: {ancestor.handoff_name} ({ancestor.model_stem})")
        else:
            # A cross-run ancestor is recorded under ancestors/ (its verdict,
            # config, fingerprint and hashes) and loaded from where it lives —
            # never copied into this run.
            record_ancestor(RUN_DIR, ancestor)
            print(f"Reusing certified {NODE.id!r} from run {ancestor.run_id}: {ancestor.handoff_name} ({ancestor.model_stem})")
        NODE_HANDOFF[NODE.id] = {
            "model": ancestor.model_stem,
            "vecnorm": str(ancestor.normalization_path),
            "stage_dir": ancestor.stage_dir,
            "run_dir": ancestor.source_run_dir,
            "run_id": None if same_run else ancestor.run_id,
            "model_sha256": ancestor.model_sha256,
            "reused": True,
        }
        continue

    # What the stage directory already holds decides between JUDGE and TRAIN.
    # An existing verdict the reuse rule refused is never retrained over.
    model_dir = stage_dir / "models"
    final_stem = model_dir / f"{stage_label(stage)}_final"
    final_vecnorm = f"{final_stem}_vecnorm.pkl"
    verdict = read_gate_verdict(stage_dir)
    if verdict is not None and not covered:
        if not verdict["passed"]:
            raise RuntimeError(
                f"{NODE.id!r} already holds a FAILED gate verdict in {stage_dir}: " + "; ".join(verdict["failures"])
                + ". A failed node is never silently retrained: start a new run (a fresh RUN_ID) for a new attempt."
            )
        raise RuntimeError(
            f"{NODE.id!r} holds a passed gate verdict in {stage_dir} that the reuse rule refused (the reason is "
            "printed above): its task, plant, gate or parent no longer matches this session. A changed task is a new "
            "run — mint a fresh RUN_ID. A changed gate is a re-judge, never a retrain: remove that directory's "
            "gate_verdict.json so this cell's JUDGE branch re-judges its checkpoints under this session's gate, or run "
            "scripts/backfill_gate_verdict.py --force [--gate current] on it."
        )
    if not covered and verdict is None and Path(f"{final_stem}.zip").exists() and Path(final_vecnorm).exists():
        # (3) JUDGE
        print(f"Judging {NODE.id!r}: {stage_dir} holds trained checkpoints but no gate verdict.")
        model, handoff_stem, final_model_path, handoff_vecnorm, results = evaluate_stage_checkpoints(
            stage,
            stage_dir,
            final_path=final_stem,
            final_vecnorm_path=final_vecnorm,
            timesteps=budget,
            duration_seconds=read_stage_duration(stage_dir) or 0.0,
        )
    elif not covered and verdict is None and any(model_dir.glob("*.zip")):
        raise RuntimeError(
            f"{stage_dir} holds periodic checkpoints but no completed stage ({final_stem.name}.zip is missing): an "
            f"interrupted node. Set RESUME_STAGE = {stage!r} in the RESUME cell to finish its budget, then re-run "
            "this loop; it is never retrained from scratch over them."
        )
    else:
        # (2) TRAIN
        if parent_handoff is not None:
            load_path, vecnorm_path, task_load_mode = parent_handoff["model"], parent_handoff["vecnorm"], "initialize_next_stage"
        else:
            load_path, vecnorm_path, task_load_mode = None, None, "resume_same_stage"
        # PRE-REGISTRATION for a frozen-null gate kind (recovery): thresholds and
        # null panels freeze BEFORE the policy trains, so it is never judged
        # against a moving target. The brace null is the parent's handoff held
        # at its post-settle mean action, so a root has nothing to freeze from.
        if gate_kind in FROZEN_NULL_GATE_KINDS and parent_handoff is None:
            raise RuntimeError(
                f"{NODE.id!r} is judged by the frozen-null gate {gate_kind}, whose brace null needs a parent "
                "checkpoint, but it is a root of the manifest"
            )
        if gate_kind in FROZEN_NULL_GATE_KINDS and not (stage_dir / "gate_resolution.json").exists():
            print(f"Freezing the {gate_kind} resolution for {NODE.id!r} (statue + brace null panels)...")
            freeze_recovery_gate(
                stage_dir,
                species=SPECIES,
                stage=stage,
                policy_zip=f"{parent_handoff['model']}.zip",
                vecnorm=parent_handoff["vecnorm"],
                algorithm=ALGORITHM,
            )
        if gate_kind in FROZEN_NULL_GATE_KINDS:
            # Refuse stale calibrations or abbreviated rehearsal panels before
            # spending the training budget.
            validate_recovery_resolution(
                stage_dir,
                species=SPECIES,
                stage=stage,
                policy_zip=f"{parent_handoff['model']}.zip",
                vecnorm=parent_handoff["vecnorm"],
                algorithm=ALGORITHM,
            )
        model, handoff_stem, final_model_path, _stage_dir, handoff_vecnorm, results = train_stage(
            stage=stage,
            timesteps=budget,
            run_dir=RUN_DIR,
            load_path=load_path,
            vecnorm_path=vecnorm_path,
            task_load_mode=task_load_mode,
            parent_run_id=parent_handoff["run_id"] if parent_handoff is not None else None,
            label=RUN_LABEL or None,
        )

    # A frozen-null kind rolls the trained policy over EXACTLY the frozen panel:
    # same seeds, same push schedules, same calibrated judge as the frozen
    # nulls. A missing, tampered or stale-task resolution refuses here.
    panel_successes = None
    if gate_kind in FROZEN_NULL_GATE_KINDS:
        panel_evidence = roll_policy_panel(
            stage_dir, f"{handoff_stem}.zip", handoff_vecnorm, species=SPECIES, stage=stage, algorithm=ALGORITHM
        )
        write_recovery_evidence(stage_dir, panel_evidence)
        panel_successes = panel_evidence.successes_by_seed()
        print(f"Policy panel on the frozen seeds: {sum(panel_successes.values())}/{len(panel_successes)} episodes recovered")

    # Artifacts: summary, videos, graphs — and the gate verdict, judged through
    # the one shared reporting.gates.evaluate_stage_gate, recorded onto the
    # results dict and written to gate_verdict.json (what reuse reads later).
    results = generate_stage_artifacts(
        species_cfg=SPECIES_CFG,
        stage_config=config,
        stage=stage,
        algorithm=ALGORITHM,
        stage_dir=stage_dir,
        seed=SEED,
        stage_results=results,
        recovery_successes_by_seed=panel_successes,
    )
    NODE_RESULTS[NODE.id] = results
    completed_stages.append((stage, stage_dir))
    display_stage_videos(stage, stage_dir)
    plot_training_curves([(stage, stage_dir)], STAGE_CONFIGS, ALGORITHM)
    plot_diagnostics_graphs([(stage, stage_dir)], STAGE_CONFIGS, ALGORITHM)

    # Summary and bundle BEFORE the gate check, so every certified deliverable
    # above a failure is published (result schema v4: the bundle is `partial`).
    write_training_summary(RUN_DIR, chain_results())
    save_run_bundle(chain_results(), species=SPECIES)

    # Enforce the recorded verdict. Release the runtime first: the raise halts
    # "Run all", so the auto-disconnect cell at the end would never execute.
    if not results["publication_gate_passed"]:
        _gate_msg = f"{NODE.id} failed its curriculum gate: " + "; ".join(results["gate_failures"]) + "."
        disconnect_runtime(_gate_msg)
        raise RuntimeError(_gate_msg)
    print(f"{NODE.id}: gate PASSED ({gate_kind}); handoff {handoff_stem}.zip")
    NODE_HANDOFF[NODE.id] = {
        "model": handoff_stem,
        "vecnorm": handoff_vecnorm,
        "stage_dir": stage_dir,
        "run_dir": RUN_DIR,
        "run_id": None,
        "model_sha256": sha256_file(f"{handoff_stem}.zip"),
        "reused": False,
    }

## 5b. Manual single node (debugging escape hatch)

Train ONE node outside the chain — to probe a config, or to rerun a node from an arbitrary checkpoint — by setting `MANUAL_NODE` below. The load's edge is declared, never inferred: `MANUAL_LOAD_MODE = "initialize_next_stage"` for a parent's handoff, `"resume_same_stage"` for this node's own checkpoint. The node's artifacts and verdict are generated and recorded into the summary and bundle exactly as the chain does, but the verdict is **never enforced** here (no raise, no disconnect) and the node never enters the chain's handoff table: nothing downstream trains on it. Leave `MANUAL_NODE = None` for an unattended Run-all.

In [ ]:
if not globals().get("COMMAND_TERRAIN_BEHAVIOR", False):
    # ===== MANUAL SINGLE NODE — debugging escape hatch =====
    MANUAL_NODE = None  # None = skip; a stage reference (2 or "recovery") to train ONE node outside the chain
    MANUAL_LOAD_PATH = None  # checkpoint stem to load (None = from scratch); its sidecar goes in MANUAL_VECNORM_PATH
    MANUAL_VECNORM_PATH = None
    MANUAL_LOAD_MODE = "initialize_next_stage"  # the load's declared edge: a parent's handoff, or "resume_same_stage"
    MANUAL_TIMESTEPS = None  # None = the node's TOML budget (QUICK_TEST shortens it)

    if MANUAL_NODE is not None:
        from environments.shared.curriculum import FROZEN_NULL_GATE_KINDS
        from environments.shared.harnesses.freeze_recovery_gate import (
            freeze_recovery_gate,
            roll_policy_panel,
            validate_recovery_resolution,
        )
        from environments.shared.recovery_evaluation import write_recovery_evidence
        from environments.shared.result_bundle import ResultBundleError

        _manual_entry = MANIFEST.resolve(MANUAL_NODE)
        _manual_stage = _manual_entry.reference
        _manual_config = STAGE_CONFIGS[_manual_stage]
        _manual_dir = Path(RUN_DIR) / stage_dirname(SPECIES, _manual_stage)
        _manual_gate_kind = _manual_config.get("curriculum_kwargs", {}).get("gate_kind")
        _manual_budget = MANUAL_TIMESTEPS or (50_000 if QUICK_TEST else _manual_config["curriculum_kwargs"]["timesteps"])
        _manual_policy_zip = f"{MANUAL_LOAD_PATH}.zip" if MANUAL_LOAD_PATH else None

        # A frozen-null kind (recovery) pre-registers its gate before training,
        # exactly as the chain loop does.
        if _manual_gate_kind in FROZEN_NULL_GATE_KINDS and not (_manual_dir / "gate_resolution.json").exists():
            print(f"Freezing the {_manual_gate_kind} resolution for {_manual_entry.id!r}...")
            freeze_recovery_gate(
                _manual_dir,
                species=SPECIES,
                stage=_manual_stage,
                policy_zip=_manual_policy_zip,
                vecnorm=MANUAL_VECNORM_PATH,
                algorithm=ALGORITHM,
            )
        if _manual_gate_kind in FROZEN_NULL_GATE_KINDS:
            validate_recovery_resolution(
                _manual_dir,
                species=SPECIES,
                stage=_manual_stage,
                policy_zip=_manual_policy_zip,
                vecnorm=MANUAL_VECNORM_PATH,
                algorithm=ALGORITHM,
            )

        _manual_model, _manual_handoff, _manual_final, _manual_dir, _manual_vecnorm, _manual_results = train_stage(
            stage=_manual_stage,
            timesteps=_manual_budget,
            run_dir=RUN_DIR,
            load_path=MANUAL_LOAD_PATH,
            vecnorm_path=MANUAL_VECNORM_PATH,
            task_load_mode=MANUAL_LOAD_MODE if MANUAL_LOAD_PATH else "resume_same_stage",
            label=RUN_LABEL or None,
        )

        _manual_panel = None
        if _manual_gate_kind in FROZEN_NULL_GATE_KINDS:
            _manual_evidence = roll_policy_panel(
                _manual_dir,
                f"{_manual_handoff}.zip",
                _manual_vecnorm,
                species=SPECIES,
                stage=_manual_stage,
                algorithm=ALGORITHM,
            )
            write_recovery_evidence(_manual_dir, _manual_evidence)
            _manual_panel = _manual_evidence.successes_by_seed()

        _manual_results = generate_stage_artifacts(
            species_cfg=SPECIES_CFG,
            stage_config=_manual_config,
            stage=_manual_stage,
            algorithm=ALGORITHM,
            stage_dir=_manual_dir,
            seed=SEED,
            stage_results=_manual_results,
            recovery_successes_by_seed=_manual_panel,
        )
        NODE_RESULTS[_manual_entry.id] = _manual_results
        completed_stages.append((_manual_stage, _manual_dir))
        display_stage_videos(_manual_stage, _manual_dir)

        # Recorded, never enforced: this cell is a probe, not a chain step, so no
        # raise and no disconnect — and it never feeds NODE_HANDOFF.
        if _manual_results["publication_gate_passed"]:
            print(f"{_manual_entry.id}: gate verdict PASS ({_manual_gate_kind})")
        else:
            print(f"{_manual_entry.id}: gate verdict FAIL ({_manual_gate_kind}) —", "; ".join(_manual_results["gate_failures"]))

        try:
            write_training_summary(RUN_DIR, chain_results())
            save_run_bundle(chain_results(), species=SPECIES)
        except ResultBundleError as exc:
            # Never swallowed silently (D-A11): a manual node that the bundle
            # cannot place (e.g. its parent is not in this run) is reported.
            print(f"Result bundle not rewritten for the manual node: {exc}")

## 6. Resume an Interrupted Node (Colab runtime cap)

Colab reclaims the runtime at its session cap (~24 h), and a long node does not
survive it: run `20260821_142144` lost locomotion at 5,488,640 of its 8M-step
budget — no `stage2_final`, no summary, no figures. What does survive are the
periodic checkpoints (`CheckpointCallback`, every 100k steps; that run trained
at the former 500k cadence, so its newest was `stage2_5000000_steps.zip` beside
its matched `stage2_vecnormalize_5000000_steps.pkl` — a ~488k-step / ~2.2 h
loss window the denser cadence shrinks to at most ~30 min, and retention keeps
5 step-points so storage is unchanged). This cell resumes an interrupted stage
from its newest **intact** periodic checkpoint, turning the cap from a
run-killer into a checkpoint boundary.

In a fresh runtime:

1. Run sections 1–4 (not the chain loop), but set `RUN_ID` in the storage cell (section 2) to the
   interrupted run's id (e.g. `"20260821_142144"`) so `RUN_DIR` points at the
   existing run directory instead of minting a new one.
2. Set `RESUME_STAGE` below to the interrupted node's reference — an integer
   (`2`) or a semantic id (`"recovery"`) — and run the cell.

Pointing `RUN_ID` at an old run is for resuming *that run's* interrupted node only.
A certified checkpoint from a run trained behind this checkout's policy interface (at
most `WIDEN_MAX_REVISION_GAP` revisions, default 1; decision D-C17) is brought in with
`WIDEN_FROM` (configuration cell) in a **new** `RUN_ID` —
never by pointing `RUN_ID` at the old run (the storage cell refuses to re-mint a run
directory whose `provenance.json` records another plant identity or seed, and the
chain loop refuses a verdict its reuse rule rejects with "mint a fresh RUN_ID") — and
`SEED` must equal that parent run's seed (decision D-C14; the widen cell refuses any
other value). Set `SEED` before the storage cell runs: a session refused at the widen
cell has already minted `RUN_DIR` under the wrong `training_seed`, and the storage cell
will not re-mint it under another seed — correct `SEED`, restart the runtime (or
`del _ACTIVE_RUN_ID`) so a fresh `RUN_ID` is minted, and delete the stray directory (it
holds only `provenance.json`). `N_ENVS` in a widen session describes the nodes trained
here; the widened root's run block keeps the parent's `n_envs`.

The cell globs the stage's `models/` directory for
`<stage_label>_<steps>_steps.zip` and walks the step-points newest first,
validating each candidate pair before trusting it: the matched
`<stage_label>_vecnormalize_<steps>_steps.pkl` sidecar must exist, the
checkpoint zip must pass an integrity check, and the sidecar must unpickle. A
truncated or orphaned newest pair — exactly what an ungraceful runtime reclaim
mid-write leaves behind — is skipped with a warning and the next older
step-point is used instead, so a corrupt newest pair no longer needs deleting
by hand. The first intact pair wins, and the cell trains for the remaining
budget (stage timesteps minus the chosen checkpoint's steps). The load is
validated as `resume_same_stage`, so a config edited between sessions fails
closed on the task fingerprint instead of silently resuming across a task
change. A candidate missing its VecNormalize sidecar is never resumed:
training a loaded policy under fresh normalization statistics is the
silent-collapse failure mode of review F3. Only when **no** step-point
survives validation does the cell raise, listing every skipped file and why.

Resume-from-periodic-checkpoint requires the sidecar fallback fix in
`train_base` (this branch, review F3) — before it, checkpoint loading probed
only the curated `<base>_vecnorm.pkl` name, warned, and trained under fresh
statistics silently. Afterwards, re-run the chain loop (section 5): its JUDGE
branch finds the node's final checkpoint without a gate verdict, evaluates it,
judges the gate and records the verdict, then continues down the chain. The
recorded duration accumulates across the interrupted and resumed sessions.

In [ ]:
if not globals().get("COMMAND_TERRAIN_BEHAVIOR", False):
    # ===== RESUME AN INTERRUPTED STAGE — opt-in escape hatch =====
    RESUME_STAGE = None  # None = skip; set to a stage ref (2 or "recovery") to resume

    if RESUME_STAGE is not None:
        import pickle
        import re
        import zipfile

        from environments.shared.stage_manifest import stage_dirname, stage_label

        cfg_res = STAGE_CONFIGS[RESUME_STAGE]
        # Same budget derivation as the chain loop: the remaining
        # budget must be measured against the budget the interrupted run used.
        budget_res = 50_000 if QUICK_TEST else cfg_res["curriculum_kwargs"]["timesteps"]
        label_res = stage_label(RESUME_STAGE)
        model_dir_res = RUN_DIR / stage_dirname(SPECIES, RESUME_STAGE) / "models"

        # SB3's CheckpointCallback names periodic checkpoints
        # {prefix}_{steps}_steps.zip and (save_vecnormalize=True in
        # _build_core_callbacks) writes a matched
        # {prefix}_vecnormalize_{steps}_steps.pkl sidecar.
        ckpts_res = []
        for p in sorted(model_dir_res.glob(f"{label_res}_*_steps.zip")):
            m = re.fullmatch(re.escape(label_res) + r"_(\d+)_steps", p.stem)
            if m:
                ckpts_res.append((int(m.group(1)), p))
        if not ckpts_res:
            raise RuntimeError(
                f"No periodic checkpoint {label_res}_*_steps.zip in {model_dir_res} — "
                "check that RUN_ID (section 2) names the interrupted run."
            )
        # Newest-first candidate walk with integrity validation: the newest pair
        # is exactly the file an ungraceful runtime reclaim may have left
        # truncated or orphaned mid-write, so a bad newest candidate falls back
        # to the next older step-point instead of killing the resume.
        steps_res = ckpt_res = vecnorm_ckpt_res = None
        skipped_res = []
        for cand_steps, cand_ckpt in sorted(ckpts_res, key=lambda sc: sc[0], reverse=True):
            cand_vecnorm = model_dir_res / f"{label_res}_vecnormalize_{cand_steps}_steps.pkl"
            if not cand_vecnorm.exists():
                # Training a loaded policy under fresh normalization statistics is
                # the silent-collapse failure mode of review F3 — skip the
                # orphaned checkpoint rather than degrade.
                reason = f"{cand_ckpt.name}: missing matched VecNormalize sidecar {cand_vecnorm.name}"
                print(f"WARNING: skipping {reason}")
                skipped_res.append(reason)
                continue
            try:
                with zipfile.ZipFile(cand_ckpt) as zf_res:
                    bad_member_res = zf_res.testzip()
                    names_res = zf_res.namelist()
                if bad_member_res is not None:
                    raise zipfile.BadZipFile(f"corrupt archive member {bad_member_res!r}")
                # A truncated SB3 checkpoint can still open: zipfile locks onto a
                # nested torch archive's end-of-directory record, so testzip alone
                # passes. Require SB3's own members in the OUTER archive.
                if "data" not in names_res or not any(n.endswith("policy.pth") for n in names_res):
                    raise zipfile.BadZipFile(
                        "outer archive lacks SB3 members (truncated checkpoint; "
                        f"found {sorted(names_res)[:5]}...)"
                    )
            except Exception as exc:
                reason = f"{cand_ckpt.name}: bad/truncated checkpoint zip ({exc})"
                print(f"WARNING: skipping {reason}")
                skipped_res.append(reason)
                continue
            try:
                with open(cand_vecnorm, "rb") as f:
                    pickle.load(f)
            except Exception as exc:
                reason = f"{cand_vecnorm.name}: VecNormalize sidecar does not unpickle ({exc})"
                print(f"WARNING: skipping {reason}")
                skipped_res.append(reason)
                continue
            steps_res, ckpt_res, vecnorm_ckpt_res = cand_steps, cand_ckpt, cand_vecnorm
            break
        if ckpt_res is None:
            raise FileNotFoundError(
                f"No intact periodic checkpoint pair {label_res}_<steps>_steps.zip + "
                f"{label_res}_vecnormalize_<steps>_steps.pkl in {model_dir_res}. Skipped: "
                + "; ".join(skipped_res)
                + ". Deleting a corrupt newest pair by hand is no longer needed — this "
                "cell already fell back through every older step-point."
            )
        remaining_res = max(budget_res - steps_res, 0)
        print(f"Newest intact periodic checkpoint: {ckpt_res}")
        print(f"Matched VecNormalize:              {vecnorm_ckpt_res}")
        if skipped_res:
            print(f"({len(skipped_res)} newer candidate(s) skipped as incomplete/corrupt — see warnings above)")
        print(f"Checkpoint steps: {steps_res:,} of {budget_res:,} — remaining budget: {remaining_res:,}")

        if remaining_res == 0:
            print(
                "Stage budget already spent — nothing to resume; re-run the chain loop (section 5): its JUDGE "
                "branch evaluates and gates the existing checkpoints."
            )
        else:
            model_res, path_res, final_path_res, dir_res, vecnorm_res, results_res = train_stage(
                stage=RESUME_STAGE,
                timesteps=remaining_res,
                load_path=str(ckpt_res),
                run_dir=RUN_DIR,
                vecnorm_path=str(vecnorm_ckpt_res),
                task_load_mode="resume_same_stage",
                label=RUN_LABEL or None,
            )

## 7. Evaluate the Behavior's Policy

In [ ]:
if not globals().get("COMMAND_TERRAIN_BEHAVIOR", False):
    # Evaluate BEHAVIOR's deliverable — the target node's certified handoff checkpoint —
    # using the library's evaluate() function, which provides full locomotion metrics
    # (gait symmetry, cost of transport, stride frequency, velocity consistency, etc.)
    # via LocomotionMetrics.
    if TARGET_NODE.id not in NODE_HANDOFF:
        raise RuntimeError(
            f"{TARGET_NODE.id!r} (behavior {BEHAVIOR!r}) is not certified in this session; run the chain loop "
            "(section 5) to the end first"
        )
    print(f"Evaluating the {BEHAVIOR!r} policy — {TARGET_NODE.id} ({ALGORITHM})...")
    evaluate(
        species_cfg=SPECIES_CFG,
        stage_configs=STAGE_CONFIGS,
        model_path=NODE_HANDOFF[TARGET_NODE.id]["model"] + ".zip",
        n_episodes=30,
        render=False,
        stage=TARGET_NODE.reference,
        algorithm=ALGORITHM.lower(),
    )

## 8. Training Curves

In [ ]:
if not globals().get("COMMAND_TERRAIN_BEHAVIOR", False):
    # Re-plot the graphs of every node trained or judged in THIS run (convenient for
    # re-running this cell standalone). Reused ancestors keep their curves in the run
    # that trained them.
    for stage_ref, stage_dir in completed_stages:
        print(f"\n{'=' * 40}")
        print(f"{MANIFEST.resolve(stage_ref).id}: {STAGE_CONFIGS[stage_ref]['name']}")
        print(f"{'=' * 40}")
        plot_training_curves(
            [(stage_ref, stage_dir)], STAGE_CONFIGS, ALGORITHM, save_path=Path(stage_dir) / "training_curves.png"
        )
        plot_diagnostics_graphs([(stage_ref, stage_dir)], STAGE_CONFIGS, ALGORITHM, save_dir=stage_dir)

## 9. Replay the Chain's Videos

Display the saved videos for every node of the behavior's chain — a reused
ancestor plays from the run that certified it. Videos are recorded by
`generate_stage_artifacts` after each node completes.

In [ ]:
if not globals().get("COMMAND_TERRAIN_BEHAVIOR", False):
    for entry in CHAIN:
        handoff = NODE_HANDOFF.get(entry.id)
        print(f"\n{'=' * 40}")
        if handoff is None:
            print(f"{entry.id}: not certified in this session (no videos)")
            continue
        origin = f"reused from run {handoff['run_id']}" if handoff["run_id"] else "this run"
        print(f"{entry.id}: {STAGE_CONFIGS[entry.reference]['name']} ({origin})")
        print(f"{'=' * 40}")
        display_stage_videos(entry.reference, handoff["stage_dir"])

## Saved behavior videos and terrain maps

Behavior sessions display their scored videos and matching maps here. Re-run this cell to view saved evidence without training again.


In [ ]:
# Direction and terrain training; the canonical curriculum uses the existing cells below.
# ===== DISPLAY SAVED BEHAVIOR EVIDENCE =====
if COMMAND_TERRAIN_BEHAVIOR:
    from environments.shared.behavior_notebook import display_notebook_behavior

    display_notebook_behavior(BEHAVIOR_PLAN.output_dir)


## 10. Cleanup

In [ ]:
if not globals().get("COMMAND_TERRAIN_BEHAVIOR", False):
    # Verify the bundle without rewriting immutable artifacts. Every deliverable
    # this run certified is published (result schema v4); the bundle is
    # `complete` only when its target — BEHAVIOR's node — and every recorded
    # deliverable are certified. A chain that stopped above its target (or a
    # manual node that failed its gate) leaves the bundle publishable but
    # `partial`: report which deliverable is uncertified instead of failing.
    from environments.shared.result_bundle import ResultBundleError

    try:
        bundle_report = validate_result_bundle(RUN_DIR, require_complete=True)
    except ResultBundleError as incomplete:
        bundle_report = validate_result_bundle(RUN_DIR, require_complete=False, require_publishable=True)
        _uncertified = sorted(key for key, record in bundle_report["deliverables"].items() if not record["certified"])
        print(f"Result bundle is publishable but not complete ({bundle_report['status']}): uncertified {_uncertified}")
        print(f"  {incomplete}")
    print(f"Result bundle verified: {bundle_report['status']}")

    print("Training complete!")
    print(f"\nBehavior: {BEHAVIOR} -> {TARGET_NODE.id}")
    print(f"Algorithm: {ALGORITHM}")
    print(f"Run directory: {RUN_DIR}")
    for entry in CHAIN:
        handoff = NODE_HANDOFF.get(entry.id)
        if handoff is None:
            print(f"  {entry.id}: NOT certified in this session")
        elif handoff["run_id"]:
            print(f"  {entry.id}: reused from run {handoff['run_id']} — {handoff['model']}.zip")
        else:
            print(f"  {entry.id}: certified in this run — {handoff['model']}.zip")
    print("\nTo run the other algorithm, change ALGORITHM at the top and re-run all cells.")

## 11. Auto-Disconnect

Releases the Colab runtime when the behavior chain finishes so it doesn't sit idle burning GPU credits. A node that fails its gate disconnects the same way from within the chain loop (the gate's `RuntimeError` halts "Run all" before this cell). Set `AUTO_DISCONNECT = False` in the configuration cell to keep the runtime alive for interactive work.

In [ ]:
if not globals().get("COMMAND_TERRAIN_BEHAVIOR", False):
    disconnect_runtime(f"Training finished — behavior {BEHAVIOR!r} ({TARGET_NODE.id}) chain complete.")

In [ ]:
if COMMAND_TERRAIN_BEHAVIOR:
    disconnect_runtime(f"Behavior {BEHAVIOR!r} session {BEHAVIOR_RESULT['status']}; artifacts saved.")
